<a href="https://colab.research.google.com/github/EngrOluchi/Aufgaben/blob/main/DT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# STAGE 1 — CELL 1
# SMAC EV digital twin: central configuration.
# No dynamic simulation, PINN training or external data upload.

import math
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 1. INSTALLED MOTOR: SUPPLIED MANUFACTURER SPECIFICATIONS
# ============================================================

motor_specs = {
    "manufacturer": "Benevelli",
    "model": "SMAC 200-035",
    "machine_description": "three-phase synchronous motor",
    "inverter_supply_V": 48.0,

    "rated_power_W": 5500.0,
    "nominal_speed_rpm": 3000.0,
    "nominal_torque_Nm": 17.5,
    "nominal_phase_current_Arms": 170.0,
    "rated_duty": "S2-60 min",

    "peak_torque_Nm": 31.5,
    "peak_phase_current_Arms": 340.0,
    "peak_duty": "S2-5 min",

    "instant_torque_Nm": 38.0,
    "instant_phase_current_Arms": 460.0,
    "instant_duration_s": None,

    "Kt_Nm_per_Arms": 0.10,
    "Ke_Vrms_per_krpm": 6.6,
    "rotor_inertia_kgm2": 0.006,
    "pole_count": 8,
    "nominal_frequency_Hz": 200.0,
    "cooling_method": "air",
    "temperature_sensor": "KTY 84-130",

    # These must not be inferred from graph-axis endpoints.
    "maximum_speed_rpm": None,
    "winding_resistance_ohm": None,
    "Ld_H": None,
    "Lq_H": None,
    "flux_linkage_Wb": None,

    "evidence": "supplied SMAC manufacturer datasheet",
}

# ============================================================
# 2. BATTERY: SPECIFICATIONS PER COMPLETE BATTERY UNIT
# ============================================================

battery_params = {
    "manufacturer": "LiTime",
    "chemistry": "prismatic LiFePO4",

    "nominal_voltage_V": 48.0,
    "nominal_capacity_Ah": 100.0,
    "nominal_energy_Wh": 4800.0,

    "bms_rating_A": 100.0,
    "continuous_discharge_A": 100.0,
    "short_discharge_A": 280.0,
    "short_discharge_duration_s": 5.0,
    "continuous_output_power_W": 4800.0,

    "continuous_charge_A": 100.0,
    "recommended_charge_A": 20.0,
    "charge_method": "CC/CV",
    "charge_voltage_V": 54.0,
    "charge_voltage_tolerance_V": 0.75,

    # Manufacturer upper bound, NOT an identified circuit R0.
    "internal_resistance_upper_bound_ohm": 0.040,

    "charge_temperature_min_C": 0.0,
    "charge_temperature_max_C": 50.0,
    "discharge_temperature_min_C": -20.0,
    "discharge_temperature_max_C": 60.0,
    "storage_temperature_min_C": -10.0,
    "storage_temperature_max_C": 50.0,

    # Unknown physical configuration and identified parameters.
    "installed_unit_count": None,
    "installed_connection": None,
    "series_cells": None,
    "parallel_strings": None,

    "R0_pack_ohm": None,
    "R1_pack_ohm": None,
    "C1_pack_F": None,
    "capacity_soh": None,
    "initial_soc": None,
    "thermal_capacity_J_per_K": None,
    "thermal_conductance_W_per_K": None,

    # Charging voltage is NOT automatically a BMS trip threshold.
    "bms_undervoltage_trip_V": None,
    "bms_overvoltage_trip_V": None,

    "evidence": "user-supplied battery specification images",
}

# ============================================================
# 3. SECONDARY VEHICLE GEOMETRY
# Thesis reference: verify against the physical converted EV.
# ============================================================

vehicle_geometry = {
    "length_m": 2.820,
    "width_m": 1.370,
    "height_m": 1.780,
    "wheelbase_m": 1.920,
    "rear_track_m": 1.140,
    "number_of_wheels": 3,
}

vehicle_reference = {
    "gross_mass_limit_kg": 645.0,
    "evidence": "secondary reference: Schurr thesis",
}

thesis_model = {
    "radius_m": 0.205,
    "frontal_area_m2": 1.60,
    "drag_coefficient": 0.60,
    "Crr": 0.015,
    "rho": 1.184,
    "wheel_mass_kg": 12.0,
    "proposed_reduction": 6.1,
    "evidence": "secondary thesis inputs; not physical calibration",
}

# ============================================================
# 4. ACTUAL VEHICLE INFORMATION STILL REQUIRED
# ============================================================

controller_specs = {
    "manufacturer": None,
    "model": None,
    "battery_current_limit_A": None,
    "motor_current_limit_Arms": None,
    "maximum_dc_voltage_V": None,
    "control_mode": None,
}

vehicle_measurements = {
    "converted_ready_mass_kg": None,
    "driver_mass_kg": None,
    "additional_payload_kg": None,
    "test_total_mass_kg": None,
    "loaded_wheel_radius_m": None,
    "reduction": None,
    "cg_x_m": None,
    "cg_height_m": None,
}

# ============================================================
# 5. EXPLICIT SIMULATION SCENARIO
# This does not establish the installed battery arrangement.
# ============================================================

assumed = {
    "battery_units_in_parallel": 1,
    "battery_units_in_series": 1,
}

environment = {
    "gravity_mps2": 9.81,
    "ambient_temperature_C": None,
    "road_grade_rad": None,
    "wind_speed_mps": None,
}

architecture = {
    "motor": "installed Benevelli SMAC 200-035",
    "battery": "LiTime 48 V 100 Ah unit",
    "controller": "installed unit; model identification pending",
    "transmission": "actual arrangement and reduction pending",
    "vehicle_layout": "single front wheel; two rear driven wheels",
}

model_settings = {
    "configuration_version": "SMAC_baseline_0.1",
    "pinn_enabled": False,
    "physical_validation_status": "not_validated",
    "battery_configuration_status": "single-unit scenario, unconfirmed",
}

# ============================================================
# 6. DERIVED QUANTITIES
# ============================================================

derived = {
    "motor_pole_pairs": motor_specs["pole_count"] // 2,

    "motor_nominal_power_check_W": (
        motor_specs["nominal_torque_Nm"]
        * motor_specs["nominal_speed_rpm"]
        * 2.0 * math.pi / 60.0
    ),

    "battery_unit_energy_kWh": (
        battery_params["nominal_voltage_V"]
        * battery_params["nominal_capacity_Ah"] / 1000.0
    ),

    "scenario_battery_voltage_V": (
        battery_params["nominal_voltage_V"]
        * assumed["battery_units_in_series"]
    ),

    "scenario_battery_capacity_Ah": (
        battery_params["nominal_capacity_Ah"]
        * assumed["battery_units_in_parallel"]
    ),
}

derived["scenario_battery_energy_kWh"] = (
    derived["scenario_battery_voltage_V"]
    * derived["scenario_battery_capacity_Ah"] / 1000.0
)

# ============================================================
# 7. CONSISTENCY CHECKS
# ============================================================

assert math.isclose(
    derived["motor_nominal_power_check_W"],
    motor_specs["rated_power_W"],
    rel_tol=0.02,
)

assert math.isclose(
    derived["battery_unit_energy_kWh"] * 1000.0,
    battery_params["nominal_energy_Wh"],
    rel_tol=1e-10,
)

assert math.isclose(
    motor_specs["nominal_speed_rpm"] / 60.0
    * derived["motor_pole_pairs"],
    motor_specs["nominal_frequency_Hz"],
)

assert (
    battery_params["short_discharge_A"]
    >= battery_params["continuous_discharge_A"]
)

assert (
    battery_params["recommended_charge_A"]
    <= battery_params["continuous_charge_A"]
)

# ============================================================
# 8. BASELINE SUMMARY
# ============================================================

parameter_register = pd.DataFrame([
    (
        "Motor rated shaft power",
        motor_specs["rated_power_W"], "W",
        "Manufacturer: S2-60 min"
    ),
    (
        "Motor nominal power calculated from torque and RPM",
        derived["motor_nominal_power_check_W"], "W",
        "Calculated consistency check"
    ),
    (
        "Battery unit nominal voltage",
        battery_params["nominal_voltage_V"], "V",
        "Supplied specification"
    ),
    (
        "Battery unit nominal capacity",
        battery_params["nominal_capacity_Ah"], "Ah",
        "Supplied specification"
    ),
    (
        "Battery unit nominal energy",
        derived["battery_unit_energy_kWh"], "kWh",
        "Calculated from supplied specification"
    ),
    (
        "Battery continuous discharge limit",
        battery_params["continuous_discharge_A"], "A DC",
        "Supplied specification"
    ),
    (
        "Wheelbase",
        vehicle_geometry["wheelbase_m"], "m",
        "Secondary thesis reference"
    ),
    (
        "Installed controller model",
        "Unknown", "—",
        "Requires identification"
    ),
    (
        "Actual total test mass",
        "Unknown", "kg",
        "Requires measurement"
    ),
    (
        "Actual transmission reduction",
        "Unknown", "—",
        "Requires verification"
    ),
], columns=["Parameter", "Value", "Unit", "Evidence"])

display(parameter_register)

print("PASS: motor torque-speed-power consistency.")
print("PASS: battery nominal-energy consistency.")
print("PASS: motor speed-pole-frequency consistency.")
print("PASS: battery current-rating ordering.")
print("Configuration established; no physical validation claimed.")
print("Actual battery unit count and arrangement remain unconfirmed.")

,Parameter,Value,Unit,Evidence
0,Motor rated shaft power,5500.0,W,Manufacturer: S2-60 min
1,Motor nominal power calculated from torque and...,5497.787144,W,Calculated consistency check
2,Battery unit nominal voltage,48.0,V,Supplied specification
3,Battery unit nominal capacity,100.0,Ah,Supplied specification
4,Battery unit nominal energy,4.8,kWh,Calculated from supplied specification
5,Battery continuous discharge limit,100.0,A DC,Supplied specification
6,Wheelbase,1.92,m,Secondary thesis reference
7,Installed controller model,Unknown,—,Requires identification
8,Actual total test mass,Unknown,kg,Requires measurement
9,Actual transmission reduction,Unknown,—,Requires verification


PASS: motor torque-speed-power consistency.
PASS: battery nominal-energy consistency.
PASS: motor speed-pole-frequency consistency.
PASS: battery current-rating ordering.
Configuration established; no physical validation claimed.
Actual battery unit count and arrangement remain unconfirmed.


In [6]:
# STAGE 1 — CELL 2
# Import and audit the physical controller driving log.
# Raw values are preserved. No parameter fitting or PINN training.

required_objects = [
    "motor_specs", "battery_params", "model_settings",
    "np", "pd", "display"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the new SMAC baseline Cell 1 first. Missing: "
        + ", ".join(missing_objects)
    )

import io
import csv
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files

# ------------------------------------------------------------
# 1. UPLOAD ONE ORIGINAL CSV
# Upload remains in the Colab session; no sharing link is created.
# ------------------------------------------------------------

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Upload exactly one controller CSV file."
    )

driving_filename, driving_bytes = next(iter(uploaded.items()))

try:
    driving_text = driving_bytes.decode("utf-8-sig")
except UnicodeDecodeError:
    # Fallback for an export using a Western Windows encoding.
    driving_text = driving_bytes.decode("cp1252")

driving_lines = driving_text.splitlines()

# Locate the actual table header instead of assuming a fixed row.
header_candidates = [
    index
    for index, line in enumerate(driving_lines)
    if line.split(";")[0].strip().strip('"') == "Time"
]

if len(header_candidates) != 1:
    raise ValueError(
        "Could not identify exactly one controller-data header."
    )

header_index = header_candidates[0]

# Check field counts before parsing.
csv_rows = [
    row
    for row in csv.reader(
        driving_lines[header_index:],
        delimiter=";"
    )
    if any(field.strip() for field in row)
]

expected_field_count = len(csv_rows[0])

bad_rows = [
    index + header_index + 1
    for index, row in enumerate(csv_rows)
    if len(row) != expected_field_count
]

if bad_rows:
    raise ValueError(
        "Inconsistent CSV field counts. Source line numbers: "
        + str(bad_rows[:10])
    )

# Preserve original strings, including decimal commas and On/Off.
driving_raw = pd.read_csv(
    io.StringIO("\n".join(driving_lines[header_index:])),
    sep=";",
    dtype=str,
    keep_default_na=False,
)

driving_raw.columns = driving_raw.columns.str.strip()

if driving_raw.columns.duplicated().any():
    raise ValueError("Duplicate column names detected.")

# ------------------------------------------------------------
# 2. PARSE TIME
# ------------------------------------------------------------

timestamps = pd.to_datetime(
    driving_raw["Time"].str.strip(),
    format="%H:%M:%S.%f",
    errors="coerce",
)

if timestamps.isna().any():
    raise ValueError(
        "Invalid timestamps found. Inspect the original Time column."
    )

clock_seconds = (
    timestamps.dt.hour * 3600.0
    + timestamps.dt.minute * 60.0
    + timestamps.dt.second
    + timestamps.dt.microsecond / 1e6
).to_numpy()

clock_changes = np.diff(clock_seconds)

# Handle an ordinary midnight crossing.
# Smaller backwards jumps remain invalid rather than being hidden.
midnight_crossings = clock_changes < -12 * 3600

day_offsets = np.concatenate([
    [0],
    np.cumsum(midnight_crossings)
]) * 86400.0

unwrapped_seconds = clock_seconds + day_offsets
elapsed_time_s = unwrapped_seconds - unwrapped_seconds[0]
sampling_intervals_s = np.diff(elapsed_time_s)

if np.any(sampling_intervals_s <= 0):
    raise ValueError(
        "Duplicate or out-of-order timestamps detected. "
        "Do not sort or delete them automatically."
    )

# ------------------------------------------------------------
# 3. DEFINE THE SELECTED SIGNAL DICTIONARY
# Units not established by documentation remain unconfirmed.
# ------------------------------------------------------------

signal_definitions = [
    (
        "throttle_command",
        "Inputs | Throttle Command",
        "Unconfirmed controller scale",
        "Input; direction convention unverified"
    ),
    (
        "brake_command",
        "Inputs | Brake Command",
        "Unconfirmed controller scale",
        "Input; not proof of all physical braking"
    ),
    (
        "motor_rpm",
        "Motor | Motor RPM",
        "rpm",
        "Motor speed; direction convention unverified"
    ),
    (
        "motor_tuning_rpm",
        "Motor Tuning | Motor RPM",
        "rpm",
        "Separate controller speed channel"
    ),
    (
        "controller_current_rms",
        "Controller | Current (RMS)",
        "RMS current; unit/scaling to verify",
        "Do not interpret as battery DC current"
    ),
    (
        "capacitor_voltage",
        "Battery | Capacitor Voltage",
        "Voltage; unit/scaling to verify",
        "Confirm measurement location"
    ),
    (
        "keyswitch_voltage",
        "Battery | Keyswitch Voltage",
        "Voltage; unit/scaling to verify",
        "Confirm measurement location"
    ),
    (
        "vehicle_speed_reported",
        "Vehicle | Vehicle Speed",
        "Unconfirmed",
        "May be derived rather than independently measured"
    ),
    (
        "motor_temperature_reported",
        "Motor | Temperature",
        "Unconfirmed",
        "Verify temperature units and sensor configuration"
    ),
    (
        "controller_temperature_reported",
        "Controller | Temperature",
        "Unconfirmed",
        "Verify temperature units and sensor location"
    ),
    (
        "bdi_reported",
        "Battery | BDI",
        "Unconfirmed indicator scale",
        "Not an independently verified SOC reference"
    ),
    (
        "battery_current_cutback",
        "Cutbacks | Battery Current Cutback",
        "Unconfirmed controller scale",
        "Not a measured current"
    ),
]

signal_dictionary = pd.DataFrame(
    signal_definitions,
    columns=["Signal", "Original column", "Unit status", "Meaning"]
)

missing_columns = [
    column
    for column in signal_dictionary["Original column"]
    if column not in driving_raw.columns
]

if missing_columns:
    raise ValueError(
        "Required channels are missing: "
        + ", ".join(missing_columns)
    )

# ------------------------------------------------------------
# 4. CREATE A NUMERIC WORKING TABLE
# The original driving_raw table is not modified.
# ------------------------------------------------------------

driving_data = pd.DataFrame({
    "time_s": elapsed_time_s
})

for signal, original_column, _, _ in signal_definitions:
    converted = pd.to_numeric(
        driving_raw[original_column]
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )

    driving_data[signal] = converted.to_numpy()

numeric_array = driving_data.to_numpy(dtype=float)

if not np.all(np.isfinite(numeric_array)):
    invalid_counts = (
        driving_data.isna().sum()
        + np.isinf(numeric_array).sum(axis=0)
    )
    display(invalid_counts.rename("Invalid values"))
    raise ValueError(
        "Selected channels contain missing or invalid numeric values."
    )

# ------------------------------------------------------------
# 5. DATA-QUALITY SUMMARY
# ------------------------------------------------------------

sampling_summary = pd.DataFrame([
    ("Recorded samples", len(driving_raw), "samples"),
    ("Original columns", len(driving_raw.columns), "columns"),
    ("Duration", elapsed_time_s[-1], "s"),
    ("Minimum interval", sampling_intervals_s.min(), "s"),
    ("Median interval", np.median(sampling_intervals_s), "s"),
    ("Maximum interval", sampling_intervals_s.max(), "s"),
    (
        "Approximate records per second",
        1.0 / np.median(sampling_intervals_s),
        "records/s"
    ),
], columns=["Quantity", "Value", "Unit"])

signal_summary = (
    driving_data.drop(columns="time_s")
    .agg(["min", "max", "nunique"])
    .T
    .rename(columns={"nunique": "Unique values"})
)

display(sampling_summary.round({"Value": 4}))
display(signal_dictionary)
display(signal_summary)

# ------------------------------------------------------------
# 6. PLOT RECORDED SIGNALS WITHOUT CHANGING THEIR SIGNS
# ------------------------------------------------------------

driving_figure = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        "Recorded throttle and brake commands",
        "Two recorded motor-speed channels",
        "Controller RMS-current channel — not battery DC current",
        "Recorded voltage channels",
        "Controller-reported vehicle speed — unit unconfirmed",
    ],
)

plot_groups = [
    (1, [
        ("throttle_command", "Throttle command"),
        ("brake_command", "Brake command"),
    ]),
    (2, [
        ("motor_rpm", "Main motor RPM"),
        ("motor_tuning_rpm", "Motor-tuning RPM"),
    ]),
    (3, [
        ("controller_current_rms", "Controller RMS current"),
    ]),
    (4, [
        ("capacitor_voltage", "Capacitor voltage"),
        ("keyswitch_voltage", "Keyswitch voltage"),
    ]),
    (5, [
        ("vehicle_speed_reported", "Reported vehicle speed"),
    ]),
]

for row_number, channels in plot_groups:
    for column, label in channels:
        driving_figure.add_trace(
            go.Scatter(
                x=driving_data["time_s"],
                y=driving_data[column],
                mode="lines+markers",
                marker=dict(size=3),
                name=label,
            ),
            row=row_number,
            col=1,
        )

driving_figure.update_xaxes(
    title_text="Elapsed recorded time [s]",
    row=5, col=1
)

driving_figure.update_layout(
    title="Physical EV driving log | original channel signs preserved",
    template="plotly_white",
    height=1150,
    hovermode="x unified",
)

driving_figure.show()

print("PASS: original CSV imported without modifying raw values.")
print("PASS: timestamps are valid and strictly increasing.")
print("PASS: selected channels contain finite numeric values.")
print("No resampling, smoothing or parameter fitting performed.")
print("Use the recorded timestamps; do not assume 100 ms sampling.")

Saving 12_11_2026_zweitefahr_kaan_100ms.CSV to 12_11_2026_zweitefahr_kaan_100ms (1).CSV


,Quantity,Value,Unit
0,Recorded samples,197.0000,samples
1,Original columns,126.0000,columns
2,Duration,161.4630,s
3,Minimum interval,0.6950,s
4,Median interval,0.7720,s
5,Maximum interval,1.4180,s
6,Approximate records per second,1.2953,records/s


,Signal,Original column,Unit status,Meaning
0,throttle_command,Inputs | Throttle Command,Unconfirmed controller scale,Input; direction convention unverified
1,brake_command,Inputs | Brake Command,Unconfirmed controller scale,Input; not proof of all physical braking
2,motor_rpm,Motor | Motor RPM,rpm,Motor speed; direction convention unverified
3,motor_tuning_rpm,Motor Tuning | Motor RPM,rpm,Separate controller speed channel
4,controller_current_rms,Controller | Current (RMS),RMS current; unit/scaling to verify,Do not interpret as battery DC current
5,capacitor_voltage,Battery | Capacitor Voltage,Voltage; unit/scaling to verify,Confirm measurement location
6,keyswitch_voltage,Battery | Keyswitch Voltage,Voltage; unit/scaling to verify,Confirm measurement location
7,vehicle_speed_reported,Vehicle | Vehicle Speed,Unconfirmed,May be derived rather than independently measured
8,motor_temperature_reported,Motor | Temperature,Unconfirmed,Verify temperature units and sensor configuration
9,controller_temperature_reported,Controller | Temperature,Unconfirmed,Verify temperature units and sensor location


,min,max,Unique values
throttle_command,-100.0,0.0,120.0
brake_command,0.0,0.0,1.0
motor_rpm,-2094.0,1.0,171.0
motor_tuning_rpm,-2173.0,0.0,162.0
controller_current_rms,1.1,268.2,169.0
capacitor_voltage,46.5,49.4,26.0
keyswitch_voltage,46.6,49.5,25.0
vehicle_speed_reported,-23.4,0.0,103.0
motor_temperature_reported,19.0,21.0,3.0
controller_temperature_reported,28.0,31.0,4.0


PASS: original CSV imported without modifying raw values.
PASS: timestamps are valid and strictly increasing.
PASS: selected channels contain finite numeric values.
No resampling, smoothing or parameter fitting performed.
Use the recorded timestamps; do not assume 100 ms sampling.


In [7]:
# STAGE 1 — CELL 3
# Controller-channel consistency checks.
# No sign reversal, filtering, fitting or parameter identification.

required_objects = [
    "driving_data", "signal_dictionary",
    "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the new SMAC notebook Cells 1 and 2 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

required_columns = [
    "time_s",
    "motor_rpm",
    "motor_tuning_rpm",
    "vehicle_speed_reported",
    "controller_current_rms",
    "battery_current_cutback",
    "motor_temperature_reported",
    "controller_temperature_reported",
]

missing_columns = [
    name for name in required_columns
    if name not in driving_data.columns
]

if missing_columns:
    raise RuntimeError(
        "Required channels are missing. Rerun Cell 2. Missing: "
        + ", ".join(missing_columns)
    )

# Independent analysis table; original imported tables stay unchanged.
channel_review = driving_data.copy()

if not np.all(np.isfinite(
    channel_review[required_columns].to_numpy(dtype=float)
)):
    raise ValueError("Selected channels contain invalid values.")

if not np.all(np.diff(channel_review["time_s"]) > 0):
    raise ValueError("Time must be strictly increasing.")

# ------------------------------------------------------------
# 1. RPM-CHANNEL DISAGREEMENT
# ------------------------------------------------------------

channel_review["rpm_difference"] = (
    channel_review["motor_rpm"]
    - channel_review["motor_tuning_rpm"]
)

rpm_difference = channel_review["rpm_difference"].to_numpy()

rpm_bias = float(np.mean(rpm_difference))
rpm_mae = float(np.mean(np.abs(rpm_difference)))
rpm_rmse = float(np.sqrt(np.mean(np.square(rpm_difference))))
rpm_max_difference = float(np.max(np.abs(rpm_difference)))

rpm_review_summary = pd.DataFrame([
    ("Mean difference: main minus tuning", rpm_bias, "rpm"),
    ("Mean absolute disagreement", rpm_mae, "rpm"),
    ("RMS disagreement", rpm_rmse, "rpm"),
    ("Maximum absolute disagreement", rpm_max_difference, "rpm"),
], columns=["Quantity", "Value", "Unit"])

print("RPM-channel comparison — neither channel is assumed to be truth:")
display(rpm_review_summary.round(3))

print("Five records with the largest RPM disagreements:")
worst_rpm_indices = (
    channel_review["rpm_difference"].abs().nlargest(5).index
)

display(
    channel_review.loc[
        worst_rpm_indices,
        [
            "time_s", "motor_rpm",
            "motor_tuning_rpm", "rpm_difference"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 2. TIMESTAMP SPACING
# ------------------------------------------------------------

time_s = channel_review["time_s"].to_numpy()
interval_s = np.diff(time_s)

sampling_review = pd.DataFrame({
    "interval_start_s": time_s[:-1],
    "interval_end_s": time_s[1:],
    "interval_s": interval_s,
})

print("Five longest recorded intervals:")
display(
    sampling_review.nlargest(5, "interval_s").round(4)
)

# ------------------------------------------------------------
# 3. CUTBACK-CHANNEL REVIEW
# Treat the scale as unconfirmed, not as current in amperes.
# ------------------------------------------------------------

cutback = channel_review["battery_current_cutback"]
cutback_changed = cutback.ne(cutback.shift())

# Exclude the initial record from change-event counting.
cutback_changed.iloc[0] = False

cutback_review = channel_review.loc[
    cutback_changed,
    [
        "time_s",
        "battery_current_cutback",
        "controller_current_rms",
        "motor_rpm",
    ]
]

print("Changes in the battery-current-cutback channel:")
display(cutback_review.round(3))

print(
    "Records with cutback-channel value below 100:",
    int((cutback < 100).sum())
)
print(
    "This count is descriptive. The meaning of 100 and lower values "
    "must be confirmed from the actual controller documentation."
)

# ------------------------------------------------------------
# 4. RECORDED TEMPERATURE SUMMARY
# ------------------------------------------------------------

temperature_review = pd.DataFrame([
    {
        "Channel": column,
        "First reported value": channel_review[column].iloc[0],
        "Last reported value": channel_review[column].iloc[-1],
        "Minimum": channel_review[column].min(),
        "Maximum": channel_review[column].max(),
        "Unique values": channel_review[column].nunique(),
    }
    for column in [
        "motor_temperature_reported",
        "controller_temperature_reported",
    ]
])

print("Temperature-channel summary — unit/scaling still unconfirmed:")
display(temperature_review)

# ------------------------------------------------------------
# 5. DIAGNOSTIC PLOTS
# ------------------------------------------------------------

channel_review_figure = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[
        "Recorded motor-speed channels",
        "Main RPM minus motor-tuning RPM",
        "Recorded timestamp intervals",
        "Battery-current-cutback channel: scale unconfirmed",
    ],
)

for column, label in [
    ("motor_rpm", "Main motor RPM"),
    ("motor_tuning_rpm", "Motor-tuning RPM"),
]:
    channel_review_figure.add_trace(
        go.Scatter(
            x=time_s,
            y=channel_review[column],
            name=label,
            mode="lines+markers",
            marker=dict(size=3),
        ),
        row=1, col=1,
    )

channel_review_figure.add_trace(
    go.Scatter(
        x=time_s,
        y=channel_review["rpm_difference"],
        name="RPM difference",
    ),
    row=2, col=1,
)

channel_review_figure.add_trace(
    go.Scatter(
        x=time_s[1:],
        y=interval_s,
        name="Recorded interval",
        mode="lines+markers",
        marker=dict(size=3),
    ),
    row=3, col=1,
)

channel_review_figure.add_trace(
    go.Scatter(
        x=time_s,
        y=cutback,
        name="Cutback channel",
        mode="lines+markers",
        marker=dict(size=3),
    ),
    row=4, col=1,
)

for row_number, label in [
    (1, "rpm"),
    (2, "rpm"),
    (3, "Seconds"),
    (4, "Reported value"),
]:
    channel_review_figure.update_yaxes(
        title_text=label, row=row_number, col=1
    )

channel_review_figure.update_xaxes(
    title_text="Elapsed recorded time [s]",
    row=4, col=1,
)

channel_review_figure.update_layout(
    title="Physical driving log | channel-consistency review",
    height=1050,
    template="plotly_white",
    hovermode="x unified",
)

channel_review_figure.show()

speed_rpm_figure = go.Figure()

speed_rpm_figure.add_trace(go.Scatter(
    x=channel_review["motor_rpm"],
    y=channel_review["vehicle_speed_reported"],
    mode="markers",
    marker=dict(
        size=6,
        color=time_s,
        colorscale="Viridis",
        colorbar=dict(title="Time [s]"),
    ),
    customdata=time_s,
    hovertemplate=(
        "Motor: %{x:.1f} rpm"
        "<br>Reported speed: %{y:.3f}"
        "<br>Time: %{customdata:.3f} s"
        "<extra></extra>"
    ),
    name="Recorded samples",
))

speed_rpm_figure.update_layout(
    title=(
        "Controller-reported speed versus motor RPM"
        "<br><sup>Speed unit and independence remain unverified; "
        "this is not a transmission-ratio estimate</sup>"
    ),
    xaxis_title="Main motor speed [rpm]",
    yaxis_title="Reported vehicle speed [unit unconfirmed]",
    height=450,
    template="plotly_white",
)

speed_rpm_figure.show()

print("PASS: required channels are present and finite.")
print("PASS: recorded time is strictly increasing.")
print("Raw and imported measurement tables were not modified.")
print("No physical parameter or fault diagnosis has been inferred.")

RPM-channel comparison — neither channel is assumed to be truth:


,Quantity,Value,Unit
0,Mean difference: main minus tuning,0.426,rpm
1,Mean absolute disagreement,76.355,rpm
2,RMS disagreement,107.620,rpm
3,Maximum absolute disagreement,330.000,rpm


Five records with the largest RPM disagreements:


,time_s,motor_rpm,motor_tuning_rpm,rpm_difference
183,150.178,-1241,-1571,330
175,143.780,-939,-615,-324
174,142.890,-1280,-963,-317
182,149.229,-914,-1221,307
117,95.532,-1432,-1717,285


Five longest recorded intervals:


,interval_start_s,interval_end_s,interval_s
113,91.816,93.234,1.418
80,64.155,65.537,1.382
163,134.107,135.412,1.305
188,154.301,155.571,1.270
90,72.687,73.941,1.254


Changes in the battery-current-cutback channel:


,time_s,battery_current_cutback,controller_current_rms,motor_rpm
116,94.822,94,253.2,-1204
117,95.532,76,226.0,-1432
118,96.305,64,200.3,-1683
119,97.126,57,189.6,-1842
120,97.858,52,182.5,-1949
121,98.647,53,173.6,-2047
122,99.595,100,3.2,-2094
183,150.178,69,247.7,-1241
184,151.234,59,208.8,-1577
185,151.952,60,196.5,-1780


Records with cutback-channel value below 100: 10
This count is descriptive. The meaning of 100 and lower values must be confirmed from the actual controller documentation.
Temperature-channel summary — unit/scaling still unconfirmed:


,Channel,First reported value,Last reported value,Minimum,Maximum,Unique values
0,motor_temperature_reported,19,21,19,21,3
1,controller_temperature_reported,28,30,28,31,4


PASS: required channels are present and finite.
PASS: recorded time is strictly increasing.
Raw and imported measurement tables were not modified.
No physical parameter or fault diagnosis has been inferred.


In [8]:
# STAGE 1 — CELL 4
# 3D SMAC/LiTime layout with physical-log playback.
# Packaging geometry is assumed.
# No dynamic simulation, parameter fitting or PINN training.

required_objects = [
    "vehicle_geometry", "thesis_model",
    "motor_specs", "battery_params", "assumed",
    "driving_data", "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the new SMAC notebook Cells 1–3 first. Missing: "
        + ", ".join(missing_objects)
    )

import ipywidgets as widgets
from IPython.display import clear_output
from html import escape

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

# ------------------------------------------------------------
# 1. DASHBOARD DATA SNAPSHOT
# No resampling, sign changes or inferred signals.
# ------------------------------------------------------------

dashboard_columns = [
    "time_s", "motor_rpm", "controller_current_rms",
    "capacitor_voltage", "keyswitch_voltage",
    "vehicle_speed_reported", "motor_temperature_reported",
    "controller_temperature_reported",
    "battery_current_cutback",
]

missing_columns = [
    name for name in dashboard_columns
    if name not in driving_data.columns
]

if missing_columns:
    raise RuntimeError(
        "Rerun Cell 2. Missing: " + ", ".join(missing_columns)
    )

dashboard_data = (
    driving_data[dashboard_columns]
    .copy()
    .reset_index(drop=True)
)

if not np.all(np.isfinite(
    dashboard_data.to_numpy(dtype=float)
)):
    raise ValueError("Dashboard channels contain invalid values.")

# ------------------------------------------------------------
# 2. PROVISIONAL PACKAGING
# All component positions and envelope sizes below are assumed.
# Battery box is illustrative, not final CAD.
# One battery unit is displayed as the selected scenario.
# ------------------------------------------------------------

packaging = {
    "Battery": {
        "centre": [0.72, 0.0, 0.48],
        "size": [0.55, 0.50, 0.20],
        "colour": "#27ae60",
    },
    "Controller": {
        "centre": [0.05, 0.35, 0.68],
        "size": [0.30, 0.22, 0.12],
        "colour": "#f39c12",
    },
    "SMAC motor": {
        "centre": [0.0, -0.30, 0.43],
        "size": [0.28, 0.28, 0.24],
        "colour": "#e74c3c",
    },
    "Transmission / differential": {
        "centre": [0.0, 0.05, 0.35],
        "size": [0.30, 0.30, 0.22],
        "colour": "#9b59b6",
    },
}

layout_assumptions = {
    "rear_overhang_m": 0.45,
    "tyre_width_m": 0.12,
}

D = vehicle_geometry["wheelbase_m"]
E = vehicle_geometry["rear_track_m"]
L = vehicle_geometry["length_m"]
W = vehicle_geometry["width_m"]
H = vehicle_geometry["height_m"]
r = thesis_model["radius_m"]

body_rear_x_m = -layout_assumptions["rear_overhang_m"]
body_front_x_m = body_rear_x_m + L

wheel_centres = {
    "Front wheel": [D, 0.0, r],
    "Rear left wheel": [0.0, E / 2.0, r],
    "Rear right wheel": [0.0, -E / 2.0, r],
}

assert r > 0
assert body_front_x_m > D
assert np.isclose(body_front_x_m - body_rear_x_m, L)
assert E + layout_assumptions["tyre_width_m"] <= W

for centre in wheel_centres.values():
    assert np.isclose(centre[2] - r, 0.0)

# ------------------------------------------------------------
# 3. BOX-DRAWING FUNCTION
# ------------------------------------------------------------

def add_layout_box(fig, name, centre, size, colour, opacity):
    vertices = np.array([
        [-1, -1, -1], [1, -1, -1],
        [1, 1, -1], [-1, 1, -1],
        [-1, -1, 1], [1, -1, 1],
        [1, 1, 1], [-1, 1, 1],
    ], dtype=float)

    vertices = (
        vertices * np.asarray(size) / 2.0
        + np.asarray(centre)
    )

    faces = np.array([
        [0, 1, 2], [0, 2, 3],
        [4, 5, 6], [4, 6, 7],
        [0, 1, 5], [0, 5, 4],
        [1, 2, 6], [1, 6, 5],
        [2, 3, 7], [2, 7, 6],
        [3, 0, 4], [3, 4, 7],
    ])

    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=colour,
        opacity=opacity,
        name=name,
        showlegend=True,
        hovertemplate=(
            name + "<br>Assumed packaging envelope"
            "<br>x=%{x:.2f} m"
            "<br>y=%{y:.2f} m"
            "<br>z=%{z:.2f} m<extra></extra>"
        ),
    ))

# ------------------------------------------------------------
# 4. CONTROLS
# ------------------------------------------------------------

sample_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(dashboard_data) - 1,
    description="Sample:",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

unit_selector = widgets.Dropdown(
    options=["All units"] + list(packaging),
    value="All units",
    description="Highlight:",
)

view_selector = widgets.Dropdown(
    options=["Isometric", "Side", "Front", "Top"],
    value="Isometric",
    description="View:",
)

body_toggle = widgets.Checkbox(
    value=False,
    description="Body envelope",
    indent=False,
)

dashboard_metrics = widgets.HTML()
dashboard_output = widgets.Output()

camera_views = {
    "Isometric": dict(
        eye=dict(x=1.6, y=1.6, z=0.9),
        up=dict(x=0, y=0, z=1),
    ),
    "Side": dict(
        eye=dict(x=0, y=2.5, z=0),
        up=dict(x=0, y=0, z=1),
        projection=dict(type="orthographic"),
    ),
    "Front": dict(
        eye=dict(x=2.5, y=0, z=0),
        up=dict(x=0, y=0, z=1),
        projection=dict(type="orthographic"),
    ),
    "Top": dict(
        eye=dict(x=0, y=0, z=2.5),
        up=dict(x=1, y=0, z=0),
        projection=dict(type="orthographic"),
    ),
}

# ------------------------------------------------------------
# 5. REFRESH DASHBOARD
# ------------------------------------------------------------

def refresh_recorded_dashboard(change=None):
    record = dashboard_data.iloc[sample_slider.value]
    fig = go.Figure()

    # Wheel outlines.
    angle = np.linspace(0.0, 2.0 * np.pi, 100)

    for name, (x0, y0, z0) in wheel_centres.items():
        for side_index, side in enumerate([-0.5, 0.5]):
            fig.add_trace(go.Scatter3d(
                x=x0 + r * np.cos(angle),
                y=np.full_like(
                    angle,
                    y0 + side * layout_assumptions["tyre_width_m"]
                ),
                z=z0 + r * np.sin(angle),
                mode="lines",
                line=dict(color="#252525", width=6),
                name=name,
                legendgroup=name,
                showlegend=(side_index == 0),
            ))

    # Rear axle: location schematic only.
    fig.add_trace(go.Scatter3d(
        x=[0, 0], y=[-E / 2, E / 2], z=[r, r],
        mode="lines",
        line=dict(color="#7f8c8d", width=7),
        name="Rear axle",
    ))

    add_layout_box(
        fig, "Chassis envelope",
        [(body_rear_x_m + body_front_x_m) / 2, 0, 0.29],
        [L, 0.85, 0.08], "#607d8b", 0.20,
    )

    for name, unit in packaging.items():
        opacity = (
            0.90 if unit_selector.value in ["All units", name]
            else 0.10
        )

        add_layout_box(
            fig, name, unit["centre"], unit["size"],
            unit["colour"], opacity,
        )

        fig.add_trace(go.Scatter3d(
            x=[unit["centre"][0]],
            y=[unit["centre"][1]],
            z=[
                unit["centre"][2]
                + unit["size"][2] / 2 + 0.08
            ],
            mode="text",
            text=[name],
            textfont=dict(size=11, color="#172b4d"),
            showlegend=False,
            hoverinfo="skip",
        ))

    # Illustrative steering-column location.
    fig.add_trace(go.Scatter3d(
        x=[D, D - 0.25],
        y=[0, 0],
        z=[r + 0.08, 1.15],
        mode="lines",
        line=dict(color="#00a6c8", width=6),
        name="Steering — assumed geometry",
    ))

    if body_toggle.value:
        add_layout_box(
            fig, "Body envelope",
            [
                (body_rear_x_m + body_front_x_m) / 2,
                0, (0.35 + H) / 2
            ],
            [L, W, H - 0.35],
            "#3498db", 0.07,
        )

    fig.update_layout(
        title=(
            "SMAC EV | physical-log playback"
            f"<br><sup>Sample {sample_slider.value} | "
            f"elapsed time {record['time_s']:.3f} s"
            " | assumed packaging geometry</sup>"
        ),
        scene=dict(
            xaxis_title="Forward x [m]",
            yaxis_title="Left y [m]",
            zaxis_title="Height z [m]",
            aspectmode="data",
            camera=camera_views[view_selector.value],
            zaxis=dict(range=[0, H + 0.15]),
        ),
        height=600,
        margin=dict(l=0, r=0, t=80, b=0),
    )

    metric_rows = [
        ("Motor speed", f"{record['motor_rpm']:.0f} rpm"),
        (
            "Controller RMS current — scaling unconfirmed",
            f"{record['controller_current_rms']:.1f}"
        ),
        (
            "Capacitor-voltage channel — scaling unconfirmed",
            f"{record['capacitor_voltage']:.2f}"
        ),
        (
            "Reported vehicle speed — unit unconfirmed",
            f"{record['vehicle_speed_reported']:.2f}"
        ),
        (
            "Reported motor temperature — unit unconfirmed",
            f"{record['motor_temperature_reported']:.1f}"
        ),
        (
            "Reported controller temperature — unit unconfirmed",
            f"{record['controller_temperature_reported']:.1f}"
        ),
        (
            "Battery-current-cutback — scale unconfirmed",
            f"{record['battery_current_cutback']:.1f}"
        ),
        ("Battery DC current", "Not available in this log"),
        ("Battery temperature", "Not available in this log"),
        ("Verified battery SOC", "Not available"),
        ("Motor shaft torque", "Not measured"),
    ]

    table_rows = "".join(
        "<tr>"
        f"<td style='padding:5px'>{escape(label)}</td>"
        f"<td style='padding:5px'><b>{escape(value)}</b></td>"
        "</tr>"
        for label, value in metric_rows
    )

    dashboard_metrics.value = (
        "<div style='font-family:Arial;padding:12px;"
        "background:#f4f7fb;border:1px solid #d5deea;"
        "border-radius:8px'>"
        "<h3>Recorded channels — no simulated operating signals</h3>"
        f"<p>Elapsed time: <b>{record['time_s']:.3f} s</b></p>"
        "<table style='width:100%;text-align:left'>"
        + table_rows
        + "</table>"
        "<p>One battery unit is illustrated as a scenario. "
        "Actual installed count remains unconfirmed. "
        "Motor and battery DC currents are not interchangeable.</p>"
        "</div>"
    )

    with dashboard_output:
        clear_output(wait=True)
        fig.show(config={"displaylogo": False})

for control in [
    sample_slider, unit_selector, view_selector, body_toggle
]:
    control.observe(refresh_recorded_dashboard, names="value")

display(widgets.VBox([
    widgets.HBox(
        [unit_selector, view_selector, body_toggle],
        layout=widgets.Layout(flex_flow="row wrap"),
    ),
    sample_slider,
    dashboard_metrics,
    dashboard_output,
]))

refresh_recorded_dashboard()

print("3D layout and recorded-channel playback initialized.")
print("Move the slider between stored samples.")
print("Sample spacing is irregular; sample number is not elapsed seconds.")
print("Geometry is assumed; no physical clearance validation performed.")

3D layout and recorded-channel playback initialized.
Move the slider between stored samples.
Sample spacing is irregular; sample number is not elapsed seconds.
Geometry is assumed; no physical clearance validation performed.


In [9]:
# CELL 4A — REPAIR THE 3D DASHBOARD DISPLAY
# Run after the existing Cell 4.

required_objects = [
    "refresh_recorded_dashboard",
    "dashboard_data", "dashboard_metrics", "dashboard_output",
    "sample_slider", "unit_selector", "view_selector", "body_toggle",
    "packaging", "layout_assumptions", "wheel_centres",
    "add_layout_box", "camera_views",
    "D", "E", "L", "W", "H", "r",
    "body_rear_x_m", "body_front_x_m",
    "np", "go", "display", "clear_output",
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Cells 1–4 first. Missing: "
        + ", ".join(missing_objects)
    )

from html import escape

# Remove the currently registered refresh callback.
# This avoids two dashboards refreshing after one control change.
for control in [
    sample_slider, unit_selector, view_selector, body_toggle
]:
    control.unobserve(
        refresh_recorded_dashboard,
        names="value"
    )


def refresh_recorded_dashboard(change=None):
    record = dashboard_data.iloc[sample_slider.value]
    selected_unit = unit_selector.value

    fig = go.Figure()
    angle = np.linspace(0.0, 2.0 * np.pi, 100)

    # --------------------------------------------------------
    # 1. EXISTING WHEEL GEOMETRY
    # --------------------------------------------------------

    for name, (x0, y0, z0) in wheel_centres.items():
        for side_index, side in enumerate([-0.5, 0.5]):
            fig.add_trace(go.Scatter3d(
                x=x0 + r * np.cos(angle),
                y=np.full_like(
                    angle,
                    y0 + side
                    * layout_assumptions["tyre_width_m"]
                ),
                z=z0 + r * np.sin(angle),
                mode="lines",
                line=dict(color="#252525", width=7),
                name=name,
                legendgroup=name,
                showlegend=(side_index == 0),
            ))

    fig.add_trace(go.Scatter3d(
        x=[0, 0],
        y=[-E / 2, E / 2],
        z=[r, r],
        mode="lines",
        line=dict(color="#7f8c8d", width=7),
        name="Rear axle",
    ))

    add_layout_box(
        fig,
        "Chassis envelope",
        [(body_rear_x_m + body_front_x_m) / 2, 0, 0.29],
        [L, 0.85, 0.08],
        "#607d8b",
        0.20,
    )

    # --------------------------------------------------------
    # 2. COMPONENTS — NO OVERLAPPING PERMANENT LABELS
    # Hover still identifies every component.
    # --------------------------------------------------------

    for name, unit in packaging.items():
        opacity = (
            0.95
            if selected_unit in ["All units", name]
            else 0.12
        )

        add_layout_box(
            fig,
            name,
            unit["centre"],
            unit["size"],
            unit["colour"],
            opacity,
        )

    # One selected-component label with a leader line.
    if selected_unit in packaging:
        unit = packaging[selected_unit]
        x0, y0, z0 = unit["centre"]
        component_top_z = z0 + unit["size"][2] / 2

        label_position = [x0, y0, H + 0.12]

        fig.add_trace(go.Scatter3d(
            x=[x0, label_position[0]],
            y=[y0, label_position[1]],
            z=[component_top_z, label_position[2]],
            mode="lines+markers",
            line=dict(color=unit["colour"], width=4),
            marker=dict(size=3, color=unit["colour"]),
            showlegend=False,
            hoverinfo="skip",
        ))

        fig.add_trace(go.Scatter3d(
            x=[label_position[0]],
            y=[label_position[1]],
            z=[label_position[2]],
            mode="text",
            text=[selected_unit],
            textposition="top center",
            textfont=dict(size=14, color="#172b4d"),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=[D, D - 0.25],
        y=[0, 0],
        z=[r + 0.08, 1.15],
        mode="lines",
        line=dict(color="#00a6c8", width=6),
        name="Steering — schematic",
    ))

    if body_toggle.value:
        add_layout_box(
            fig,
            "Body envelope",
            [
                (body_rear_x_m + body_front_x_m) / 2,
                0,
                (0.35 + H) / 2,
            ],
            [L, W, H - 0.35],
            "#3498db",
            0.07,
        )

    # --------------------------------------------------------
    # 3. CLEARER CAMERA AREA AND BOTTOM LEGEND
    # --------------------------------------------------------

    fig.update_layout(
        title=dict(
            text=(
                "SMAC EV — recorded-data playback"
                f"<br><sup>Elapsed time: {record['time_s']:.3f} s"
                " | component geometry and placement assumed</sup>"
            ),
            x=0.02,
            y=0.98,
            font=dict(size=17),
        ),
        scene=dict(
            domain=dict(x=[0, 1], y=[0.12, 0.90]),
            xaxis_title="Forward x [m]",
            yaxis_title="Left y [m]",
            zaxis_title="Height z [m]",
            aspectmode="data",
            camera=camera_views[view_selector.value],
            xaxis=dict(
                range=[body_rear_x_m - 0.15, body_front_x_m + 0.15]
            ),
            yaxis=dict(range=[-W / 2 - 0.10, W / 2 + 0.10]),
            zaxis=dict(range=[-0.03, H + 0.40]),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=-0.02,
            yanchor="top",
            font=dict(size=10),
            bgcolor="rgba(255,255,255,0.95)",
        ),
        height=850,
        autosize=True,
        margin=dict(l=15, r=15, t=85, b=160),
        paper_bgcolor="white",
    )

    # --------------------------------------------------------
    # 4. UPDATE RECORDED INDICATORS
    # --------------------------------------------------------

    metric_rows = [
        ("Motor speed", f"{record['motor_rpm']:.0f} rpm"),
        (
            "Controller RMS current — scaling unconfirmed",
            f"{record['controller_current_rms']:.1f}",
        ),
        (
            "Capacitor voltage — scaling unconfirmed",
            f"{record['capacitor_voltage']:.2f}",
        ),
        (
            "Reported vehicle speed — unit unconfirmed",
            f"{record['vehicle_speed_reported']:.2f}",
        ),
        (
            "Motor temperature — unit unconfirmed",
            f"{record['motor_temperature_reported']:.1f}",
        ),
        (
            "Controller temperature — unit unconfirmed",
            f"{record['controller_temperature_reported']:.1f}",
        ),
        (
            "Battery-current-cutback — scale unconfirmed",
            f"{record['battery_current_cutback']:.1f}",
        ),
        ("Battery DC current / SOC / temperature", "Unavailable"),
        ("Motor shaft torque", "Not measured"),
    ]

    table_rows = "".join(
        "<tr>"
        f"<td style='padding:5px'>{escape(label)}</td>"
        f"<td style='padding:5px'><b>{escape(value)}</b></td>"
        "</tr>"
        for label, value in metric_rows
    )

    dashboard_metrics.value = (
        "<div style='font-family:Arial;padding:12px;"
        "background:#f4f7fb;border:1px solid #d5deea;"
        "border-radius:8px'>"
        f"<h3>Recorded sample {sample_slider.value}</h3>"
        f"<p>Elapsed time: {record['time_s']:.3f} s</p>"
        "<table style='width:100%;text-align:left'>"
        + table_rows
        + "</table>"
        "<p>Select one component to display its label. "
        "Hover over other units to identify them. "
        "One battery unit is illustrated; installed count unconfirmed."
        "</p></div>"
    )

    with dashboard_output:
        clear_output(wait=True)
        fig.show(config={
            "displaylogo": False,
            "responsive": True,
        })


# Register only the repaired callback.
for control in [
    sample_slider, unit_selector, view_selector, body_toggle
]:
    control.observe(
        refresh_recorded_dashboard,
        names="value"
    )

refresh_recorded_dashboard()

print("Display repair applied to the existing dashboard.")
print("Select SMAC motor or Battery to see a single elevated label.")
print("Use Top or Front to distinguish the two rear wheels.")

Display repair applied to the existing dashboard.
Select SMAC motor or Battery to see a single elevated label.
Use Top or Front to distinguish the two rear wheels.


In [10]:
# CELL 4A — REPAIR THE 3D DASHBOARD DISPLAY
# Run after the existing Cell 4.

required_objects = [
    "refresh_recorded_dashboard",
    "dashboard_data", "dashboard_metrics", "dashboard_output",
    "sample_slider", "unit_selector", "view_selector", "body_toggle",
    "packaging", "layout_assumptions", "wheel_centres",
    "add_layout_box", "camera_views",
    "D", "E", "L", "W", "H", "r",
    "body_rear_x_m", "body_front_x_m",
    "np", "go", "display", "clear_output",
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Cells 1–4 first. Missing: "
        + ", ".join(missing_objects)
    )

from html import escape

# Remove the currently registered refresh callback.
# This avoids two dashboards refreshing after one control change.
for control in [
    sample_slider, unit_selector, view_selector, body_toggle
]:
    control.unobserve(
        refresh_recorded_dashboard,
        names="value"
    )


def refresh_recorded_dashboard(change=None):
    record = dashboard_data.iloc[sample_slider.value]
    selected_unit = unit_selector.value

    fig = go.Figure()
    angle = np.linspace(0.0, 2.0 * np.pi, 100)

    # --------------------------------------------------------
    # 1. EXISTING WHEEL GEOMETRY
    # --------------------------------------------------------

    for name, (x0, y0, z0) in wheel_centres.items():
        for side_index, side in enumerate([-0.5, 0.5]):
            fig.add_trace(go.Scatter3d(
                x=x0 + r * np.cos(angle),
                y=np.full_like(
                    angle,
                    y0 + side
                    * layout_assumptions["tyre_width_m"]
                ),
                z=z0 + r * np.sin(angle),
                mode="lines",
                line=dict(color="#252525", width=7),
                name=name,
                legendgroup=name,
                showlegend=(side_index == 0),
            ))

    fig.add_trace(go.Scatter3d(
        x=[0, 0],
        y=[-E / 2, E / 2],
        z=[r, r],
        mode="lines",
        line=dict(color="#7f8c8d", width=7),
        name="Rear axle",
    ))

    add_layout_box(
        fig,
        "Chassis envelope",
        [(body_rear_x_m + body_front_x_m) / 2, 0, 0.29],
        [L, 0.85, 0.08],
        "#607d8b",
        0.20,
    )

    # --------------------------------------------------------
    # 2. COMPONENTS — NO OVERLAPPING PERMANENT LABELS
    # Hover still identifies every component.
    # --------------------------------------------------------

    for name, unit in packaging.items():
        opacity = (
            0.95
            if selected_unit in ["All units", name]
            else 0.12
        )

        add_layout_box(
            fig,
            name,
            unit["centre"],
            unit["size"],
            unit["colour"],
            opacity,
        )

    # One selected-component label with a leader line.
    if selected_unit in packaging:
        unit = packaging[selected_unit]
        x0, y0, z0 = unit["centre"]
        component_top_z = z0 + unit["size"][2] / 2

        label_position = [x0, y0, H + 0.12]

        fig.add_trace(go.Scatter3d(
            x=[x0, label_position[0]],
            y=[y0, label_position[1]],
            z=[component_top_z, label_position[2]],
            mode="lines+markers",
            line=dict(color=unit["colour"], width=4),
            marker=dict(size=3, color=unit["colour"]),
            showlegend=False,
            hoverinfo="skip",
        ))

        fig.add_trace(go.Scatter3d(
            x=[label_position[0]],
            y=[label_position[1]],
            z=[label_position[2]],
            mode="text",
            text=[selected_unit],
            textposition="top center",
            textfont=dict(size=14, color="#172b4d"),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Scatter3d(
        x=[D, D - 0.25],
        y=[0, 0],
        z=[r + 0.08, 1.15],
        mode="lines",
        line=dict(color="#00a6c8", width=6),
        name="Steering — schematic",
    ))

    if body_toggle.value:
        add_layout_box(
            fig,
            "Body envelope",
            [
                (body_rear_x_m + body_front_x_m) / 2,
                0,
                (0.35 + H) / 2,
            ],
            [L, W, H - 0.35],
            "#3498db",
            0.07,
        )

    # --------------------------------------------------------
    # 3. CLEARER CAMERA AREA AND BOTTOM LEGEND
    # --------------------------------------------------------

    fig.update_layout(
        title=dict(
            text=(
                "SMAC EV — recorded-data playback"
                f"<br><sup>Elapsed time: {record['time_s']:.3f} s"
                " | component geometry and placement assumed</sup>"
            ),
            x=0.02,
            y=0.98,
            font=dict(size=17),
        ),
        scene=dict(
            domain=dict(x=[0, 1], y=[0.12, 0.90]),
            xaxis_title="Forward x [m]",
            yaxis_title="Left y [m]",
            zaxis_title="Height z [m]",
            aspectmode="data",
            camera=camera_views[view_selector.value],
            xaxis=dict(
                range=[body_rear_x_m - 0.15, body_front_x_m + 0.15]
            ),
            yaxis=dict(range=[-W / 2 - 0.10, W / 2 + 0.10]),
            zaxis=dict(range=[-0.03, H + 0.40]),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=-0.02,
            yanchor="top",
            font=dict(size=10),
            bgcolor="rgba(255,255,255,0.95)",
        ),
        height=850,
        autosize=True,
        margin=dict(l=15, r=15, t=85, b=160),
        paper_bgcolor="white",
    )

    # --------------------------------------------------------
    # 4. UPDATE RECORDED INDICATORS
    # --------------------------------------------------------

    metric_rows = [
        ("Motor speed", f"{record['motor_rpm']:.0f} rpm"),
        (
            "Controller RMS current — scaling unconfirmed",
            f"{record['controller_current_rms']:.1f}",
        ),
        (
            "Capacitor voltage — scaling unconfirmed",
            f"{record['capacitor_voltage']:.2f}",
        ),
        (
            "Reported vehicle speed — unit unconfirmed",
            f"{record['vehicle_speed_reported']:.2f}",
        ),
        (
            "Motor temperature — unit unconfirmed",
            f"{record['motor_temperature_reported']:.1f}",
        ),
        (
            "Controller temperature — unit unconfirmed",
            f"{record['controller_temperature_reported']:.1f}",
        ),
        (
            "Battery-current-cutback — scale unconfirmed",
            f"{record['battery_current_cutback']:.1f}",
        ),
        ("Battery DC current / SOC / temperature", "Unavailable"),
        ("Motor shaft torque", "Not measured"),
    ]

    table_rows = "".join(
        "<tr>"
        f"<td style='padding:5px'>{escape(label)}</td>"
        f"<td style='padding:5px'><b>{escape(value)}</b></td>"
        "</tr>"
        for label, value in metric_rows
    )

    dashboard_metrics.value = (
        "<div style='font-family:Arial;padding:12px;"
        "background:#f4f7fb;border:1px solid #d5deea;"
        "border-radius:8px'>"
        f"<h3>Recorded sample {sample_slider.value}</h3>"
        f"<p>Elapsed time: {record['time_s']:.3f} s</p>"
        "<table style='width:100%;text-align:left'>"
        + table_rows
        + "</table>"
        "<p>Select one component to display its label. "
        "Hover over other units to identify them. "
        "One battery unit is illustrated; installed count unconfirmed."
        "</p></div>"
    )

    with dashboard_output:
        clear_output(wait=True)
        fig.show(config={
            "displaylogo": False,
            "responsive": True,
        })


# Register only the repaired callback.
for control in [
    sample_slider, unit_selector, view_selector, body_toggle
]:
    control.observe(
        refresh_recorded_dashboard,
        names="value"
    )

refresh_recorded_dashboard()

print("Display repair applied to the existing dashboard.")
print("Select SMAC motor or Battery to see a single elevated label.")
print("Use Top or Front to distinguish the two rear wheels.")

Display repair applied to the existing dashboard.
Select SMAC motor or Battery to see a single elevated label.
Use Top or Front to distinguish the two rear wheels.


In [11]:
# STAGE 1 — CELL 5
# Baseline review and configuration checkpoint.
# Does not modify the raw driving log or model parameters.

import json
import hashlib
from copy import deepcopy

required_objects = [
    "motor_specs",
    "battery_params",
    "vehicle_geometry",
    "vehicle_reference",
    "thesis_model",
    "controller_specs",
    "vehicle_measurements",
    "assumed",
    "environment",
    "architecture",
    "model_settings",
    "derived",
    "driving_raw",
    "driving_data",
    "driving_filename",
    "driving_bytes",
    "signal_dictionary",
    "packaging",
    "layout_assumptions",
    "np",
    "pd",
    "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the new SMAC notebook Cells 1–4 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# 1. REVIEW THE ACTIVE BASELINE
# ------------------------------------------------------------

time_values = driving_data["time_s"].to_numpy(dtype=float)

stage1_checks = [
    (
        "SMAC motor configuration selected",
        motor_specs["manufacturer"] == "Benevelli"
        and motor_specs["model"] == "SMAC 200-035",
    ),
    (
        "Motor rated power is 5500 W",
        np.isclose(motor_specs["rated_power_W"], 5500.0),
    ),
    (
        "Battery unit capacity is 100 Ah",
        np.isclose(
            battery_params["nominal_capacity_Ah"], 100.0
        ),
    ),
    (
        "Battery nominal-energy consistency",
        np.isclose(
            battery_params["nominal_voltage_V"]
            * battery_params["nominal_capacity_Ah"],
            battery_params["nominal_energy_Wh"],
        ),
    ),
    (
        "Raw and working tables have matching record counts",
        len(driving_raw) == len(driving_data),
    ),
    (
        "Recorded time is finite and strictly increasing",
        len(time_values) >= 2
        and np.all(np.isfinite(time_values))
        and np.all(np.diff(time_values) > 0),
    ),
    (
        "Original recorded channels remain available",
        "Time" in driving_raw.columns
        and "Controller | Current (RMS)" in driving_raw.columns,
    ),
    (
        "PINN training remains deferred",
        model_settings["pinn_enabled"] is False,
    ),
]

stage1_check_table = pd.DataFrame([
    {
        "Check": name,
        "Result": "PASS" if bool(passed) else "FAIL",
    }
    for name, passed in stage1_checks
])

display(stage1_check_table)

if not all(bool(passed) for _, passed in stage1_checks):
    raise RuntimeError(
        "Resolve the failed baseline checks before continuing."
    )

# ------------------------------------------------------------
# 2. REPORT IMPORTANT UNKNOWN PARAMETERS
# None means not established; it does not mean zero.
# ------------------------------------------------------------

unknown_items = [
    (
        "Installed battery count",
        battery_params["installed_unit_count"],
        "Determines total installed capacity and energy",
    ),
    (
        "Installed battery connection",
        battery_params["installed_connection"],
        "Determines system voltage and parallel arrangement",
    ),
    (
        "Controller model",
        controller_specs["model"],
        "Needed for operating limits and signal definitions",
    ),
    (
        "Controller battery-current limit",
        controller_specs["battery_current_limit_A"],
        "Must be distinguished from motor RMS current",
    ),
    (
        "Actual transmission reduction",
        vehicle_measurements["reduction"],
        "Needed for motor-to-wheel conversion",
    ),
    (
        "Total mass during physical test",
        vehicle_measurements["test_total_mass_kg"],
        "Needed for acceleration and load calculations",
    ),
    (
        "Loaded wheel radius",
        vehicle_measurements["loaded_wheel_radius_m"],
        "Needed for wheel-speed and traction calculations",
    ),
    (
        "Actual initial SOC",
        battery_params["initial_soc"],
        "Needed for physical battery replay",
    ),
    (
        "Identified battery R0",
        battery_params["R0_pack_ohm"],
        "Published resistance bound is not an identified R0",
    ),
]

unknown_register = pd.DataFrame([
    {
        "Parameter": name,
        "Status": "UNKNOWN" if value is None else "Entered",
        "Current value": value,
        "Why it matters": explanation,
    }
    for name, value, explanation in unknown_items
])

display(unknown_register)

# ------------------------------------------------------------
# 3. RECORD THE SOURCE DATASET IDENTITY
# Hash identifies the exact uploaded bytes; it does not upload them.
# ------------------------------------------------------------

recorded_data_summary = {
    "source_filename": driving_filename,
    "source_sha256": hashlib.sha256(
        driving_bytes
    ).hexdigest(),
    "record_count": int(len(driving_data)),
    "original_column_count": int(len(driving_raw.columns)),
    "duration_s": float(time_values[-1] - time_values[0]),
    "minimum_interval_s": float(np.diff(time_values).min()),
    "median_interval_s": float(np.median(np.diff(time_values))),
    "maximum_interval_s": float(np.diff(time_values).max()),
    "resampled": False,
    "channel_units_fully_verified": False,
    "battery_dc_current_available": False,
    "shaft_torque_measurement_available": False,
}

# ------------------------------------------------------------
# 4. CREATE AN INDEPENDENT BASELINE SNAPSHOT
# Raw measurement rows are not duplicated in this JSON.
# ------------------------------------------------------------

baseline_record = deepcopy({
    "baseline_version": "SMAC_LiTime_0.1",
    "status": "configuration_and_data_audit_complete",
    "physical_validation_status": "not_validated",

    "motor_specs": motor_specs,
    "battery_params": battery_params,
    "vehicle_geometry": vehicle_geometry,
    "vehicle_reference": vehicle_reference,
    "thesis_model": thesis_model,
    "controller_specs": controller_specs,
    "vehicle_measurements": vehicle_measurements,

    "assumed_scenario": assumed,
    "environment": environment,
    "architecture": architecture,
    "model_settings": model_settings,
    "derived": derived,

    "packaging": packaging,
    "layout_assumptions": layout_assumptions,

    "recorded_data_summary": recorded_data_summary,
    "signal_dictionary": signal_dictionary.to_dict(
        orient="records"
    ),

    "limitations": [
        "Installed battery count and connection unconfirmed.",
        "Actual controller configuration not yet established.",
        "Transmission ratio and physical test mass unverified.",
        "Some recorded channel units and meanings unconfirmed.",
        "3D packaging geometry and mounting positions assumed.",
        "No calibrated coupled drivetrain model yet.",
        "No PINN training or physical validation performed.",
    ],
})

def checkpoint_json_converter(value):
    """Handle NumPy values without silently converting unknown objects."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(
        "Unsupported checkpoint value: "
        + type(value).__name__
    )

checkpoint_text = json.dumps(
    baseline_record,
    indent=2,
    allow_nan=False,
    default=checkpoint_json_converter,
)

checkpoint_filename = "SMAC_LiTime_baseline_v0_1.json"

with open(
    checkpoint_filename,
    "w",
    encoding="utf-8",
) as file:
    file.write(checkpoint_text)

print("PASS: Stage 1 baseline review completed.")
print("Checkpoint created:", checkpoint_filename)
print("Find it in the Colab Files panel and download a copy.")
print("Also save the notebook and preserve the original CSV.")
print("Unknown parameters remain explicit; physical validation is pending.")

,Check,Result
0,SMAC motor configuration selected,PASS
1,Motor rated power is 5500 W,PASS
2,Battery unit capacity is 100 Ah,PASS
3,Battery nominal-energy consistency,PASS
4,Raw and working tables have matching record co...,PASS
5,Recorded time is finite and strictly increasing,PASS
6,Original recorded channels remain available,PASS
7,PINN training remains deferred,PASS


,Parameter,Status,Current value,Why it matters
0,Installed battery count,UNKNOWN,None,Determines total installed capacity and energy
1,Installed battery connection,UNKNOWN,None,Determines system voltage and parallel arrange...
2,Controller model,UNKNOWN,None,Needed for operating limits and signal definit...
3,Controller battery-current limit,UNKNOWN,None,Must be distinguished from motor RMS current
4,Actual transmission reduction,UNKNOWN,None,Needed for motor-to-wheel conversion
5,Total mass during physical test,UNKNOWN,None,Needed for acceleration and load calculations
6,Loaded wheel radius,UNKNOWN,None,Needed for wheel-speed and traction calculations
7,Actual initial SOC,UNKNOWN,None,Needed for physical battery replay
8,Identified battery R0,UNKNOWN,None,Published resistance bound is not an identifie...


PASS: Stage 1 baseline review completed.
Checkpoint created: SMAC_LiTime_baseline_v0_1.json
Find it in the Colab Files panel and download a copy.
Also save the notebook and preserve the original CSV.
Unknown parameters remain explicit; physical validation is pending.


In [12]:
# STAGE 2 — CELL 6
# Battery DC-current charge accounting.
# Manufacturer parameters and simulation assumptions remain separate.
# No voltage, thermal, BMS or PINN model yet.

required_objects = [
    "battery_params",
    "assumed",
    "stage1_checks",
    "np",
    "pd",
    "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the new SMAC notebook Cells 1–5 first. Missing: "
        + ", ".join(missing_objects)
    )

if not all(bool(passed) for _, passed in stage1_checks):
    raise RuntimeError(
        "Resolve Stage 1 baseline failures before continuing."
    )

import plotly.graph_objects as go

# ------------------------------------------------------------
# 1. EXPLICIT MODEL ASSUMPTIONS
# Do not overwrite unknown physical SOC or SOH in battery_params.
# ------------------------------------------------------------

battery_model_assumptions = {
    "initial_soc": 0.80,
    "available_capacity_fraction": 1.0,
    "charge_accounting_efficiency": 1.0,
    "self_discharge_modelled": False,
    "evidence": "illustrative simulation assumptions",
}

# Existing unit-level scenario from Cell 1.
parallel_units = assumed["battery_units_in_parallel"]
series_units = assumed["battery_units_in_series"]

if not (
    isinstance(parallel_units, (int, np.integer))
    and isinstance(series_units, (int, np.integer))
    and parallel_units > 0
    and series_units > 0
):
    raise ValueError("Battery-unit counts must be positive integers.")

# This step implements only the existing one-unit scenario.
# It does not imply that series connection is manufacturer-approved.
if parallel_units != 1 or series_units != 1:
    raise ValueError(
        "This first battery model uses one complete battery unit. "
        "Confirm the installed arrangement before extending it."
    )

battery_derived = {
    "scenario_nominal_voltage_V": (
        battery_params["nominal_voltage_V"]
    ),
    "scenario_nominal_capacity_Ah": (
        battery_params["nominal_capacity_Ah"]
    ),
}

battery_derived["available_capacity_Ah"] = (
    battery_derived["scenario_nominal_capacity_Ah"]
    * battery_model_assumptions["available_capacity_fraction"]
)

# ------------------------------------------------------------
# 2. SOC SIMULATION FUNCTION
# Current at sample k applies until sample k+1.
# The final current sample does not change the final SOC.
# ------------------------------------------------------------

def simulate_battery_soc(time_s, battery_current_A):
    """
    Ideal battery charge accounting under an imposed DC-current profile.

    Positive current = discharge.
    Negative current = charge.

    Current is held constant over each recorded interval.
    Tests outside 0–100% SOC are rejected rather than clipped.

    No current limits, voltage limits or BMS action are enforced here.
    """
    time_s = np.asarray(time_s, dtype=float)
    battery_current_A = np.asarray(
        battery_current_A, dtype=float
    )

    if (
        time_s.ndim != 1
        or battery_current_A.ndim != 1
        or time_s.shape != battery_current_A.shape
        or len(time_s) < 2
    ):
        raise ValueError(
            "Time and battery current must be matching 1D arrays "
            "with at least two samples."
        )

    if not (
        np.all(np.isfinite(time_s))
        and np.all(np.isfinite(battery_current_A))
        and np.all(np.diff(time_s) > 0)
    ):
        raise ValueError(
            "Inputs must be finite and time must strictly increase."
        )

    capacity_fraction = float(
        battery_model_assumptions["available_capacity_fraction"]
    )
    initial_soc = float(
        battery_model_assumptions["initial_soc"]
    )

    if not (
        np.isfinite(capacity_fraction)
        and 0 < capacity_fraction <= 1
        and np.isfinite(initial_soc)
        and 0 <= initial_soc <= 1
    ):
        raise ValueError("Invalid capacity fraction or initial SOC.")

    available_capacity_Ah = (
        battery_params["nominal_capacity_Ah"]
        * capacity_fraction
    )

    if not (
        np.isfinite(available_capacity_Ah)
        and available_capacity_Ah > 0
    ):
        raise ValueError("Available capacity must be positive.")

    interval_charge_Ah = (
        battery_current_A[:-1]
        * np.diff(time_s)
        / 3600.0
    )

    net_charge_removed_Ah = np.concatenate([
        [0.0],
        np.cumsum(interval_charge_Ah),
    ])

    soc = (
        initial_soc
        - net_charge_removed_Ah / available_capacity_Ah
    )

    if np.any(soc < -1e-12) or np.any(soc > 1.0 + 1e-12):
        raise ValueError(
            "Requested test exceeds 0–100% SOC. "
            "Reduce current or duration. BMS control is not modelled."
        )

    return pd.DataFrame({
        "time_s": time_s,
        "battery_current_A": battery_current_A,
        "SOC": soc,
        "net_charge_removed_Ah": net_charge_removed_Ah,
    })

# ------------------------------------------------------------
# 3. CONTROLLED DISCHARGE TEST
# ------------------------------------------------------------

battery_test_time_s = np.arange(0.0, 601.0, 1.0)
battery_test_current_A = np.full_like(
    battery_test_time_s, 30.0
)

battery_soc_result = simulate_battery_soc(
    battery_test_time_s,
    battery_test_current_A,
)

# ------------------------------------------------------------
# 4. MATHEMATICAL CHECKS
# ------------------------------------------------------------

expected_charge_removed_Ah = 30.0 * 600.0 / 3600.0

expected_final_soc = (
    battery_model_assumptions["initial_soc"]
    - expected_charge_removed_Ah
    / battery_derived["available_capacity_Ah"]
)

assert np.isclose(
    battery_soc_result["net_charge_removed_Ah"].iloc[-1],
    expected_charge_removed_Ah,
)

assert np.isclose(
    battery_soc_result["SOC"].iloc[-1],
    expected_final_soc,
)

# Zero-current test.
battery_rest_check = simulate_battery_soc(
    battery_test_time_s,
    np.zeros_like(battery_test_time_s),
)

assert np.allclose(
    battery_rest_check["SOC"],
    battery_model_assumptions["initial_soc"],
)

# Negative current increases SOC.
# This is a sign check, not permission to charge physical hardware.
battery_charge_check = simulate_battery_soc(
    np.array([0.0, 60.0]),
    np.array([-10.0, -10.0]),
)

assert (
    battery_charge_check["SOC"].iloc[-1]
    > battery_model_assumptions["initial_soc"]
)

# Irregular-time test: 30 A for 120 s removes exactly 1 Ah.
battery_irregular_check = simulate_battery_soc(
    np.array([0.0, 0.7, 2.0, 9.5, 120.0]),
    np.full(5, 30.0),
)

assert np.isclose(
    battery_irregular_check["net_charge_removed_Ah"].iloc[-1],
    1.0,
)

# ------------------------------------------------------------
# 5. SUMMARY AND GRAPH
# ------------------------------------------------------------

battery_summary = pd.DataFrame([
    (
        "Nominal capacity per unit",
        battery_params["nominal_capacity_Ah"], "Ah",
        "Supplied specification"
    ),
    (
        "Available capacity in this test",
        battery_derived["available_capacity_Ah"], "Ah",
        "Assumed fraction of nominal capacity"
    ),
    (
        "Initial SOC",
        battery_model_assumptions["initial_soc"] * 100, "%",
        "Assumed initial state"
    ),
    (
        "Charge removed",
        battery_soc_result["net_charge_removed_Ah"].iloc[-1],
        "Ah", "Calculated"
    ),
    (
        "Final SOC",
        battery_soc_result["SOC"].iloc[-1] * 100, "%",
        "Simulated"
    ),
], columns=["Quantity", "Value", "Unit", "Evidence"])

display(battery_summary.round({"Value": 4}))

battery_soc_figure = go.Figure()

battery_soc_figure.add_trace(go.Scatter(
    x=battery_soc_result["time_s"] / 60.0,
    y=battery_soc_result["SOC"] * 100.0,
    mode="lines",
    name="Simulated SOC",
    line=dict(color="#27ae60", width=3),
))

battery_soc_figure.update_layout(
    title="Single-unit battery scenario | 30 A discharge for 10 min",
    xaxis_title="Time [min]",
    yaxis_title="SOC [%]",
    template="plotly_white",
    height=420,
)

battery_soc_figure.show()

print("PASS: charge removed matches the analytical calculation.")
print("PASS: final SOC matches the analytical calculation.")
print("PASS: zero-current SOC remains constant.")
print("PASS: negative current increases SOC.")
print("PASS: irregular sampling is handled correctly.")
print("This is a synthetic test, not replay of measured battery current.")

,Quantity,Value,Unit,Evidence
0,Nominal capacity per unit,100.0,Ah,Supplied specification
1,Available capacity in this test,100.0,Ah,Assumed fraction of nominal capacity
2,Initial SOC,80.0,%,Assumed initial state
3,Charge removed,5.0,Ah,Calculated
4,Final SOC,75.0,%,Simulated


PASS: charge removed matches the analytical calculation.
PASS: final SOC matches the analytical calculation.
PASS: zero-current SOC remains constant.
PASS: negative current increases SOC.
PASS: irregular sampling is handled correctly.
This is a synthetic test, not replay of measured battery current.


In [13]:
# STAGE 2 — CELL 7
# Resistance-only battery voltage, power and heat generation.
# Uses pack-level assumptions; no internal cell topology is assumed.

required_objects = [
    "battery_params",
    "battery_model_assumptions",
    "battery_soc_result",
    "np",
    "pd",
    "go",
    "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current SMAC notebook through Cell 6 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# 1. ASSUMED MODEL PARAMETERS
# Manufacturer specifications in battery_params are not overwritten.
# Existing assumption values are retained when this cell is rerun.
# ------------------------------------------------------------

battery_model_assumptions.setdefault("R0_pack_ohm", 0.012)

battery_model_assumptions.setdefault(
    "ocv_soc_points",
    [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0],
)

battery_model_assumptions.setdefault(
    "ocv_pack_points_V",
    [43.50, 47.25, 48.30, 49.05, 49.50, 49.95, 50.40, 51.00],
)

battery_model_assumptions["electrical_parameter_evidence"] = (
    "Synthetic pack-level OCV curve and assumed constant resistance; "
    "not identified from physical battery tests"
)

# ------------------------------------------------------------
# 2. PACK-LEVEL OCV FUNCTION
# No extrapolation outside the supplied SOC domain.
# ------------------------------------------------------------

def battery_ocv_pack(soc):
    soc = np.asarray(soc, dtype=float)

    soc_points = np.asarray(
        battery_model_assumptions["ocv_soc_points"],
        dtype=float,
    )

    voltage_points = np.asarray(
        battery_model_assumptions["ocv_pack_points_V"],
        dtype=float,
    )

    if (
        soc_points.ndim != 1
        or voltage_points.ndim != 1
        or len(soc_points) < 2
        or len(soc_points) != len(voltage_points)
    ):
        raise ValueError("Invalid OCV table dimensions.")

    if not (
        np.all(np.isfinite(soc_points))
        and np.all(np.isfinite(voltage_points))
        and np.all(np.diff(soc_points) > 0)
        and np.all(voltage_points > 0)
        and soc_points[0] >= 0
        and soc_points[-1] <= 1
    ):
        raise ValueError("Invalid OCV table values.")

    if not np.all(np.isfinite(soc)):
        raise ValueError("SOC must contain finite values.")

    if (
        np.any(soc < soc_points[0])
        or np.any(soc > soc_points[-1])
    ):
        raise ValueError(
            "SOC lies outside the OCV-table domain."
        )

    return np.interp(soc, soc_points, voltage_points)

# ------------------------------------------------------------
# 3. INSTANTANEOUS ELECTRICAL CALCULATION
# ------------------------------------------------------------

def calculate_battery_electrical(soc, battery_current_A):
    """
    Resistance-only circuit.

    Positive current: discharge.
    Negative current: charge.

    No polarization, temperature feedback or BMS action.
    """
    soc, battery_current_A = np.broadcast_arrays(
        np.asarray(soc, dtype=float),
        np.asarray(battery_current_A, dtype=float),
    )

    if not np.all(np.isfinite(battery_current_A)):
        raise ValueError("Battery current must be finite.")

    R0 = float(battery_model_assumptions["R0_pack_ohm"])

    if not np.isfinite(R0) or R0 < 0:
        raise ValueError(
            "Resistance must be finite and nonnegative."
        )

    ocv_V = battery_ocv_pack(soc)

    terminal_voltage_V = (
        ocv_V - battery_current_A * R0
    )

    if not (
        np.all(np.isfinite(terminal_voltage_V))
        and np.all(terminal_voltage_V > 0)
    ):
        raise ValueError(
            "Invalid terminal voltage. Check the imposed current "
            "and assumed resistance."
        )

    terminal_power_W = (
        terminal_voltage_V * battery_current_A
    )

    ohmic_heat_W = (
        np.square(battery_current_A) * R0
    )

    ocv_power_W = ocv_V * battery_current_A

    return {
        "ocv_V": ocv_V,
        "terminal_voltage_V": terminal_voltage_V,
        "terminal_power_W": terminal_power_W,
        "ohmic_heat_W": ohmic_heat_W,
        "ocv_power_W": ocv_power_W,
    }

# ------------------------------------------------------------
# 4. APPLY TO CELL 6'S SYNTHETIC DISCHARGE
# Original SOC result is preserved.
# ------------------------------------------------------------

required_columns = ["time_s", "battery_current_A", "SOC"]

if not all(
    column in battery_soc_result.columns
    for column in required_columns
):
    raise RuntimeError(
        "SOC result has unexpected columns. Rerun current Cell 6."
    )

if not np.all(np.isfinite(
    battery_soc_result[required_columns].to_numpy(dtype=float)
)):
    raise ValueError("SOC result contains invalid values.")

battery_electrical_result = battery_soc_result.copy()

electrical_values = calculate_battery_electrical(
    battery_electrical_result["SOC"].to_numpy(),
    battery_electrical_result["battery_current_A"].to_numpy(),
)

for column, values in electrical_values.items():
    battery_electrical_result[column] = values

battery_electrical_result["power_balance_error_W"] = (
    battery_electrical_result["ocv_power_W"]
    - battery_electrical_result["terminal_power_W"]
    - battery_electrical_result["ohmic_heat_W"]
)

# ------------------------------------------------------------
# 5. ENGINEERING CHECKS
# ------------------------------------------------------------

maximum_power_error_W = float(
    battery_electrical_result["power_balance_error_W"]
    .abs()
    .max()
)

assert maximum_power_error_W <= 1e-9, (
    f"Power balance failed: {maximum_power_error_W:.6e} W"
)

test_soc = battery_model_assumptions["initial_soc"]

rest_check = calculate_battery_electrical(
    test_soc, 0.0
)
discharge_check = calculate_battery_electrical(
    test_soc, 30.0
)
charge_check = calculate_battery_electrical(
    test_soc, -30.0
)

assert np.isclose(
    rest_check["terminal_voltage_V"],
    rest_check["ocv_V"],
)

assert np.isclose(rest_check["ohmic_heat_W"], 0.0)

R0 = battery_model_assumptions["R0_pack_ohm"]

assert np.isclose(
    discharge_check["ocv_V"]
    - discharge_check["terminal_voltage_V"],
    30.0 * R0,
)

assert np.isclose(
    charge_check["terminal_voltage_V"]
    - charge_check["ocv_V"],
    30.0 * R0,
)

assert np.isclose(
    discharge_check["ohmic_heat_W"],
    charge_check["ohmic_heat_W"],
)

# ------------------------------------------------------------
# 6. RESULTS AND GRAPHS
# ------------------------------------------------------------

display(
    battery_electrical_result.iloc[[0, -1]][[
        "time_s",
        "SOC",
        "battery_current_A",
        "ocv_V",
        "terminal_voltage_V",
        "terminal_power_W",
        "ohmic_heat_W",
    ]].round(4)
)

ocv_figure = go.Figure()

ocv_figure.add_trace(go.Scatter(
    x=np.asarray(
        battery_model_assumptions["ocv_soc_points"]
    ) * 100.0,
    y=battery_model_assumptions["ocv_pack_points_V"],
    mode="lines+markers",
    name="Synthetic pack OCV",
))

ocv_figure.update_layout(
    title="Assumed pack OCV–SOC curve — not measured battery data",
    xaxis_title="SOC [%]",
    yaxis_title="Open-circuit voltage [V]",
    template="plotly_white",
    height=380,
)

ocv_figure.show()

battery_voltage_figure = go.Figure()

for column, label in [
    ("ocv_V", "Open-circuit voltage"),
    ("terminal_voltage_V", "Terminal voltage under load"),
]:
    battery_voltage_figure.add_trace(go.Scatter(
        x=battery_electrical_result["time_s"] / 60.0,
        y=battery_electrical_result[column],
        name=label,
        mode="lines",
    ))

battery_voltage_figure.update_layout(
    title="Synthetic 30 A discharge | resistance-only voltage model",
    xaxis_title="Time [min]",
    yaxis_title="Voltage [V]",
    template="plotly_white",
    height=400,
)

battery_voltage_figure.show()

print("PASS: instantaneous circuit power balance.")
print("PASS: at zero current, terminal voltage equals OCV.")
print("PASS: zero-current ohmic heating is zero.")
print("PASS: discharge and charge voltage changes follow I × R0.")
print("PASS: equal current magnitudes give equal resistor heating.")
print(f"Maximum power-balance error: {maximum_power_error_W:.3e} W")
print("Manufacturer specifications remain separate from assumptions.")
print("No physical battery calibration or BMS validation claimed.")

,time_s,SOC,battery_current_A,ocv_V,terminal_voltage_V,terminal_power_W,ohmic_heat_W
0,0.0,0.80,30.0,49.9500,49.5900,1487.700,10.8
600,600.0,0.75,30.0,49.8375,49.4775,1484.325,10.8


PASS: instantaneous circuit power balance.
PASS: at zero current, terminal voltage equals OCV.
PASS: zero-current ohmic heating is zero.
PASS: discharge and charge voltage changes follow I × R0.
PASS: equal current magnitudes give equal resistor heating.
Maximum power-balance error: 1.812e-13 W
Manufacturer specifications remain separate from assumptions.
No physical battery calibration or BMS validation claimed.


In [14]:
# STAGE 2 — CELL 8
# One-RC battery model with charge and power-balance checks.
# Uses the current SMAC/LiTime notebook naming.
# Manufacturer battery specifications are not overwritten.

required_objects = [
    "battery_params",
    "battery_model_assumptions",
    "simulate_battery_soc",
    "battery_ocv_pack",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook through Cell 7 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. ASSUMED DYNAMIC PARAMETERS
# These are not identified manufacturer parameters.
# ------------------------------------------------------------

battery_model_assumptions.setdefault("R1_pack_ohm", 0.006)
battery_model_assumptions.setdefault("C1_pack_F", 10000.0)
battery_model_assumptions.setdefault(
    "initial_polarization_V", 0.0
)

# ------------------------------------------------------------
# 2. ONE-RC SIMULATION
# ------------------------------------------------------------

def simulate_battery_one_rc(time_s, battery_current_A):
    """
    Positive current = discharge.

    current[k] applies over [time[k], time[k+1]).
    States remain continuous at a current change.
    Endpoint voltage uses the current at that endpoint.

    No temperature feedback or BMS action is implemented.
    """
    # Reuse the validated charge-accounting implementation.
    result = simulate_battery_soc(
        time_s, battery_current_A
    ).copy()

    time_s = result["time_s"].to_numpy(dtype=float)
    current_A = result["battery_current_A"].to_numpy(dtype=float)

    R0 = float(battery_model_assumptions["R0_pack_ohm"])
    R1 = float(battery_model_assumptions["R1_pack_ohm"])
    C1 = float(battery_model_assumptions["C1_pack_F"])
    initial_V1 = float(
        battery_model_assumptions["initial_polarization_V"]
    )

    if not (
        np.all(np.isfinite([R0, R1, C1, initial_V1]))
        and R0 >= 0
        and R1 > 0
        and C1 > 0
    ):
        raise ValueError("Invalid RC model parameters.")

    tau_s = R1 * C1

    polarization_V = np.empty(len(time_s), dtype=float)
    polarization_V[0] = initial_V1

    for k, dt_s in enumerate(np.diff(time_s)):
        decay = np.exp(-dt_s / tau_s)
        one_minus_decay = -np.expm1(-dt_s / tau_s)

        polarization_V[k + 1] = (
            polarization_V[k] * decay
            + current_A[k] * R1 * one_minus_decay
        )

    ocv_V = battery_ocv_pack(result["SOC"].to_numpy())

    terminal_voltage_V = (
        ocv_V - current_A * R0 - polarization_V
    )

    if not (
        np.all(np.isfinite(terminal_voltage_V))
        and np.all(terminal_voltage_V > 0)
    ):
        raise ValueError("Invalid terminal voltage.")

    # Resistor heating:
    # R0 carries pack current; R1 carries V1/R1.
    ohmic_heat_W = np.square(current_A) * R0
    polarization_heat_W = np.square(polarization_V) / R1
    resistive_heat_W = ohmic_heat_W + polarization_heat_W

    # Equivalent RC stored energy and its rate of change.
    rc_stored_energy_J = (
        0.5 * C1 * np.square(polarization_V)
    )

    rc_energy_rate_W = (
        polarization_V * current_A
        - polarization_heat_W
    )

    terminal_power_W = terminal_voltage_V * current_A
    ocv_power_W = ocv_V * current_A

    result["polarization_V"] = polarization_V
    result["ocv_V"] = ocv_V
    result["terminal_voltage_V"] = terminal_voltage_V
    result["terminal_power_W"] = terminal_power_W
    result["ohmic_heat_W"] = ohmic_heat_W
    result["polarization_heat_W"] = polarization_heat_W
    result["resistive_heat_W"] = resistive_heat_W
    result["rc_stored_energy_J"] = rc_stored_energy_J
    result["rc_energy_rate_W"] = rc_energy_rate_W
    result["ocv_power_W"] = ocv_power_W

    result["power_balance_error_W"] = (
        ocv_power_W
        - terminal_power_W
        - resistive_heat_W
        - rc_energy_rate_W
    )

    return result

# ------------------------------------------------------------
# 3. PULSE-AND-REST TEST
# ------------------------------------------------------------

pulse_time_s = np.arange(0.0, 361.0, 1.0)

pulse_current_A = np.where(
    (pulse_time_s >= 60.0) & (pulse_time_s < 180.0),
    30.0,
    0.0,
)

battery_rc_result = simulate_battery_one_rc(
    pulse_time_s,
    pulse_current_A,
)

# ------------------------------------------------------------
# 4. ANALYTICAL AND ENGINEERING CHECKS
# ------------------------------------------------------------

R1 = battery_model_assumptions["R1_pack_ohm"]
C1 = battery_model_assumptions["C1_pack_F"]
tau_s = R1 * C1

# Includes any nonzero initial polarization entered by the user.
expected_V1_at_180 = (
    battery_model_assumptions["initial_polarization_V"]
    * np.exp(-180.0 / tau_s)
    + 30.0 * R1 * (1.0 - np.exp(-120.0 / tau_s))
)

assert np.isclose(
    battery_rc_result.loc[180, "polarization_V"],
    expected_V1_at_180,
)

assert np.isclose(
    battery_rc_result.loc[360, "polarization_V"],
    expected_V1_at_180 * np.exp(-180.0 / tau_s),
)

available_capacity_Ah = (
    battery_params["nominal_capacity_Ah"]
    * battery_model_assumptions["available_capacity_fraction"]
)

expected_final_soc = (
    battery_model_assumptions["initial_soc"]
    - (30.0 * 120.0 / 3600.0) / available_capacity_Ah
)

assert np.isclose(
    battery_rc_result["SOC"].iloc[-1],
    expected_final_soc,
)

assert np.allclose(
    battery_rc_result.loc[180:, "SOC"],
    battery_rc_result.loc[180, "SOC"],
)

maximum_power_error_W = float(
    battery_rc_result["power_balance_error_W"].abs().max()
)

assert np.isfinite(maximum_power_error_W)
assert maximum_power_error_W <= 1e-9, (
    f"Power-balance error: {maximum_power_error_W:.6e} W"
)

rest_rows = battery_rc_result["battery_current_A"] == 0.0

assert np.allclose(
    battery_rc_result.loc[rest_rows, "resistive_heat_W"],
    -battery_rc_result.loc[rest_rows, "rc_energy_rate_W"],
    atol=1e-9,
)

# Exact interval updates should agree at matching timestamps
# when pulse transitions are aligned.
fine_time_s = np.arange(0.0, 360.5, 0.5)
fine_current_A = np.where(
    (fine_time_s >= 60.0) & (fine_time_s < 180.0),
    30.0,
    0.0,
)

battery_rc_refined = simulate_battery_one_rc(
    fine_time_s, fine_current_A
)

assert np.allclose(
    battery_rc_result[["SOC", "polarization_V"]].to_numpy(),
    battery_rc_refined.iloc[::2][
        ["SOC", "polarization_V"]
    ].to_numpy(),
    atol=1e-10,
)

# ------------------------------------------------------------
# 5. RESULTS
# ------------------------------------------------------------

display(
    battery_rc_result.loc[
        [0, 60, 120, 180, 240, 360],
        [
            "time_s",
            "battery_current_A",
            "SOC",
            "polarization_V",
            "terminal_voltage_V",
            "resistive_heat_W",
            "rc_energy_rate_W",
        ],
    ].round(6)
)

battery_rc_figure = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.09,
    subplot_titles=[
        "Imposed battery DC current",
        "Terminal voltage and assumed OCV",
        "Polarization voltage",
    ],
)

battery_rc_figure.add_trace(
    go.Scatter(
        x=pulse_time_s,
        y=pulse_current_A,
        name="Battery current",
        line_shape="hv",
    ),
    row=1, col=1,
)

for column, label in [
    ("terminal_voltage_V", "Terminal voltage"),
    ("ocv_V", "OCV"),
]:
    battery_rc_figure.add_trace(
        go.Scatter(
            x=pulse_time_s,
            y=battery_rc_result[column],
            name=label,
        ),
        row=2, col=1,
    )

battery_rc_figure.add_trace(
    go.Scatter(
        x=pulse_time_s,
        y=battery_rc_result["polarization_V"],
        name="Polarization voltage",
    ),
    row=3, col=1,
)

battery_rc_figure.update_yaxes(
    title_text="Current [A]", row=1, col=1
)
battery_rc_figure.update_yaxes(
    title_text="Voltage [V]", row=2, col=1
)
battery_rc_figure.update_yaxes(
    title_text="Voltage [V]", row=3, col=1
)
battery_rc_figure.update_xaxes(
    title_text="Time [s]", row=3, col=1
)

battery_rc_figure.update_layout(
    title="Synthetic battery test | assumed one-RC model",
    height=800,
    template="plotly_white",
)

battery_rc_figure.show()

print(f"RC time constant: {tau_s:.3f} s")
print(f"Maximum power-balance error: {maximum_power_error_W:.3e} W")
print("PASS: analytical pulse response.")
print("PASS: analytical rest recovery.")
print("PASS: charge accounting and constant rest SOC.")
print("PASS: circuit power balance including RC energy storage.")
print("PASS: rest heating equals RC stored-energy release.")
print("PASS: aligned timestep-refinement check.")
print("No physical parameter calibration or BMS validation claimed.")

,time_s,battery_current_A,SOC,polarization_V,terminal_voltage_V,resistive_heat_W,rc_energy_rate_W
0,0.0,0.0,0.800,0.000000,49.950000,0.000000,0.000000
60,60.0,30.0,0.800,0.000000,49.590000,10.800000,0.000000
120,120.0,30.0,0.795,0.113782,49.464968,12.957713,1.255738
180,180.0,0.0,0.790,0.155640,49.771860,4.037283,-4.037283
240,240.0,0.0,0.790,0.057257,49.870243,0.546387,-0.546387
360,360.0,0.0,0.790,0.007749,49.919751,0.010007,-0.010007


RC time constant: 60.000 s
Maximum power-balance error: 3.095e-13 W
PASS: analytical pulse response.
PASS: analytical rest recovery.
PASS: charge accounting and constant rest SOC.
PASS: circuit power balance including RC energy storage.
PASS: rest heating equals RC stored-energy release.
PASS: aligned timestep-refinement check.
No physical parameter calibration or BMS validation claimed.


In [15]:
# STAGE 2 — CELL 9
# Lumped battery temperature driven by one-RC resistor losses.
# All thermal values remain explicitly assumed.
# Electrical parameters do not yet depend on temperature.

required_objects = [
    "battery_model_assumptions",
    "battery_rc_result",
    "battery_rc_refined",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current SMAC notebook through Cell 8 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# 1. THERMAL ASSUMPTIONS
# Do not overwrite unknown physical conditions in environment
# or unidentified properties in battery_params.
# ------------------------------------------------------------

battery_model_assumptions.setdefault(
    "thermal_capacity_J_per_K", 60000.0
)
battery_model_assumptions.setdefault(
    "thermal_conductance_W_per_K", 15.0
)
battery_model_assumptions.setdefault(
    "initial_temperature_C", 25.0
)
battery_model_assumptions.setdefault(
    "ambient_temperature_C", 25.0
)

# ------------------------------------------------------------
# 2. THERMAL FUNCTION
# ------------------------------------------------------------

def simulate_battery_temperature(
    time_s,
    heat_W,
    ambient_temperature_C,
    initial_temperature_C,
):
    """
    Uniform battery temperature.

    heat_W[k] is held constant over [time_s[k], time_s[k+1]).
    Ambient temperature is constant during this experiment.

    The interval update is exact for constant heat within the
    interval. Sampling the changing RC heat introduces a
    time-discretization approximation.
    """
    time_s = np.asarray(time_s, dtype=float)
    heat_W = np.asarray(heat_W, dtype=float)

    if (
        time_s.ndim != 1
        or heat_W.ndim != 1
        or time_s.shape != heat_W.shape
        or len(time_s) < 2
    ):
        raise ValueError(
            "Time and heat must be matching 1D arrays."
        )

    if not (
        np.all(np.isfinite(time_s))
        and np.all(np.isfinite(heat_W))
        and np.all(np.diff(time_s) > 0)
        and np.all(heat_W >= 0)
    ):
        raise ValueError(
            "Time must increase; heat must be finite and nonnegative."
        )

    Cth = float(
        battery_model_assumptions["thermal_capacity_J_per_K"]
    )
    Kth = float(
        battery_model_assumptions[
            "thermal_conductance_W_per_K"
        ]
    )

    if not (
        np.all(np.isfinite([
            Cth, Kth,
            ambient_temperature_C, initial_temperature_C,
        ]))
        and Cth > 0
        and Kth >= 0
    ):
        raise ValueError("Invalid thermal parameters.")

    temperature_C = np.empty(len(time_s), dtype=float)
    temperature_C[0] = initial_temperature_C

    for k, dt_s in enumerate(np.diff(time_s)):
        if Kth == 0:
            # No heat exchange with ambient.
            temperature_C[k + 1] = (
                temperature_C[k] + heat_W[k] * dt_s / Cth
            )
        else:
            one_minus_decay = -np.expm1(
                -Kth * dt_s / Cth
            )

            temperature_C[k + 1] = (
                temperature_C[k]
                + (
                    ambient_temperature_C
                    - temperature_C[k]
                    + heat_W[k] / Kth
                ) * one_minus_decay
            )

    return temperature_C

# ------------------------------------------------------------
# 3. CHECK THE ELECTRICAL HEAT BEFORE USING IT
# ------------------------------------------------------------

required_columns = [
    "time_s",
    "battery_current_A",
    "polarization_V",
    "resistive_heat_W",
    "power_balance_error_W",
]

for column in required_columns:
    if column not in battery_rc_result.columns:
        raise RuntimeError(
            "Electrical results are incomplete. Rerun Cell 8."
        )

if not np.all(np.isfinite(
    battery_rc_result[required_columns].to_numpy(dtype=float)
)):
    raise ValueError("Electrical results contain invalid values.")

assert np.all(
    np.abs(battery_rc_result["power_balance_error_W"]) <= 1e-9
), "Resolve the electrical power-balance error in Cell 8."

expected_heat_W = (
    np.square(battery_rc_result["battery_current_A"])
    * battery_model_assumptions["R0_pack_ohm"]
    + np.square(battery_rc_result["polarization_V"])
    / battery_model_assumptions["R1_pack_ohm"]
)

assert np.allclose(
    battery_rc_result["resistive_heat_W"],
    expected_heat_W,
    atol=1e-9,
), "Electrical heat does not match the current model parameters."

# ------------------------------------------------------------
# 4. TEMPERATURE FOR THE EXISTING PULSE-AND-REST EXPERIMENT
# ------------------------------------------------------------

ambient_C = battery_model_assumptions["ambient_temperature_C"]
initial_C = battery_model_assumptions["initial_temperature_C"]

battery_thermal_result = battery_rc_result.copy()

battery_thermal_result["battery_temperature_C"] = (
    simulate_battery_temperature(
        battery_thermal_result["time_s"],
        battery_thermal_result["resistive_heat_W"],
        ambient_C,
        initial_C,
    )
)

Cth = battery_model_assumptions["thermal_capacity_J_per_K"]
Kth = battery_model_assumptions["thermal_conductance_W_per_K"]

battery_thermal_result["heat_to_ambient_W"] = (
    Kth
    * (
        battery_thermal_result["battery_temperature_C"]
        - ambient_C
    )
)

# ------------------------------------------------------------
# 5. INDEPENDENT THERMAL CHECKS
# ------------------------------------------------------------

check_time_s = np.arange(0.0, 601.0, 1.0)

# At ambient, zero heat must leave temperature unchanged.
rest_temperature = simulate_battery_temperature(
    check_time_s,
    np.zeros_like(check_time_s),
    ambient_C,
    ambient_C,
)

assert np.allclose(rest_temperature, ambient_C)

# Constant heat: compare with the analytical solution.
constant_heat_W = 100.0

constant_heat_temperature = simulate_battery_temperature(
    check_time_s,
    np.full_like(check_time_s, constant_heat_W),
    ambient_C,
    ambient_C,
)

if Kth > 0:
    analytical_temperature = (
        ambient_C
        + constant_heat_W / Kth
        * (1.0 - np.exp(-Kth * check_time_s / Cth))
    )
else:
    analytical_temperature = (
        ambient_C + constant_heat_W * check_time_s / Cth
    )

assert np.allclose(
    constant_heat_temperature,
    analytical_temperature,
    atol=1e-9,
)

# Initially warm battery with no heat input.
cooling_temperature = simulate_battery_temperature(
    check_time_s,
    np.zeros_like(check_time_s),
    ambient_C,
    ambient_C + 5.0,
)

assert np.all(cooling_temperature >= ambient_C - 1e-10)
assert np.all(np.diff(cooling_temperature) <= 1e-10)

# Numerical refinement of the sampled heat input:
# compare Cell 8's 1 s and 0.5 s electrical results.
fine_temperature = simulate_battery_temperature(
    battery_rc_refined["time_s"],
    battery_rc_refined["resistive_heat_W"],
    ambient_C,
    initial_C,
)

assert np.allclose(
    battery_rc_result["time_s"].to_numpy(),
    battery_rc_refined["time_s"].iloc[::2].to_numpy(),
), "Refined and coarse timestamps are not aligned."

thermal_refinement_error_K = float(np.max(np.abs(
    battery_thermal_result["battery_temperature_C"].to_numpy()
    - fine_temperature[::2]
)))

# Numerical screening tolerance, not physical-model accuracy.
assert thermal_refinement_error_K < 0.001, (
    "Thermal sampling sensitivity exceeds 0.001 K. "
    "Refine the electrical/thermal sampling."
)

# ------------------------------------------------------------
# 6. SUMMARY AND PLOT
# ------------------------------------------------------------

temperature_values = (
    battery_thermal_result["battery_temperature_C"].to_numpy()
)

thermal_summary = pd.DataFrame([
    ("Initial temperature", temperature_values[0], "°C"),
    ("Maximum temperature", temperature_values.max(), "°C"),
    ("Final temperature", temperature_values[-1], "°C"),
    (
        "Maximum rise above initial temperature",
        temperature_values.max() - temperature_values[0],
        "K",
    ),
    (
        "Coarse/refined maximum temperature difference",
        thermal_refinement_error_K,
        "K",
    ),
], columns=["Quantity", "Value", "Unit"])

display(thermal_summary.round({"Value": 6}))

battery_thermal_figure = go.Figure()

battery_thermal_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=battery_thermal_result["battery_temperature_C"],
    name="Simulated uniform battery temperature",
))

battery_thermal_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=np.full(len(battery_thermal_result), ambient_C),
    name="Assumed ambient temperature",
    line=dict(dash="dash"),
))

battery_thermal_figure.update_layout(
    title="Synthetic pulse test | assumed battery thermal model",
    xaxis_title="Time [s]",
    yaxis_title="Temperature [°C]",
    template="plotly_white",
    height=430,
)

battery_thermal_figure.show()

print("PASS: electrical heat consistency.")
print("PASS: zero heating at ambient gives constant temperature.")
print("PASS: constant heating matches the analytical solution.")
print("PASS: a warmer battery does not heat up without a heat source.")
print(
    "PASS: thermal sampling-refinement difference = "
    f"{thermal_refinement_error_K:.6f} K"
)
print("Thermal parameters remain assumed, not physically calibrated.")

,Quantity,Value,Unit
0,Initial temperature,25.000000,°C
1,Maximum temperature,25.026712,°C
2,Final temperature,25.026177,°C
3,Maximum rise above initial temperature,0.026712,K
4,Coarse/refined maximum temperature difference,0.000017,K


PASS: electrical heat consistency.
PASS: zero heating at ambient gives constant temperature.
PASS: constant heating matches the analytical solution.
PASS: a warmer battery does not heat up without a heat source.
PASS: thermal sampling-refinement difference = 0.000017 K
Thermal parameters remain assumed, not physically calibrated.


In [16]:
# STAGE 3 — CELL 10
# Averaged inverter electrical power flow.
# Standalone test: not yet coupled to battery or motor dynamics.
# No PWM, phase-current model, BMS enforcement or PINN training.

required_objects = [
    "battery_params",
    "np",
    "pd",
    "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook's preceding cells first. Missing: "
        + ", ".join(missing_objects)
    )

# New dictionary: assumed inverter-model parameters.
# These are not specifications of your unidentified controller.
inverter_model_assumptions = {
    "motoring_efficiency": 0.95,
    "regeneration_efficiency": 0.95,
    "enabled_idle_power_W": 10.0,
    "evidence": "illustrative averaged-model assumptions",
}

def calculate_inverter_power(
    ac_electrical_power_W,
    dc_voltage_V,
):
    """
    Averaged enabled-inverter power balance.

    AC power > 0: inverter supplies motor electrical power.
    AC power < 0: motor returns electrical power.

    DC current > 0: power drawn from DC bus.
    DC current < 0: power returned to DC bus.

    Does not enforce battery/controller operating limits.
    Does not calculate shaft torque or motor phase current.
    """
    ac_power, dc_voltage = np.broadcast_arrays(
        np.asarray(ac_electrical_power_W, dtype=float),
        np.asarray(dc_voltage_V, dtype=float),
    )

    if not (
        np.all(np.isfinite(ac_power))
        and np.all(np.isfinite(dc_voltage))
        and np.all(dc_voltage > 0)
    ):
        raise ValueError(
            "Power must be finite and DC voltage must be positive."
        )

    eta_mot = float(
        inverter_model_assumptions["motoring_efficiency"]
    )
    eta_regen = float(
        inverter_model_assumptions["regeneration_efficiency"]
    )
    idle_W = float(
        inverter_model_assumptions["enabled_idle_power_W"]
    )

    if not (
        np.all(np.isfinite([eta_mot, eta_regen, idle_W]))
        and 0 < eta_mot <= 1
        and 0 < eta_regen <= 1
        and idle_W >= 0
    ):
        raise ValueError("Invalid inverter assumptions.")

    dc_power_W = np.where(
        ac_power >= 0,
        ac_power / eta_mot + idle_W,
        eta_regen * ac_power + idle_W,
    )

    inverter_loss_W = dc_power_W - ac_power
    dc_current_A = dc_power_W / dc_voltage

    return {
        "ac_electrical_power_W": ac_power,
        "dc_voltage_V": dc_voltage,
        "dc_power_W": dc_power_W,
        "dc_current_A": dc_current_A,
        "inverter_loss_W": inverter_loss_W,
    }

# ------------------------------------------------------------
# 1. STANDALONE TEST CASES
# Includes one deliberately demanding case for limit screening.
# ------------------------------------------------------------

test_ac_power_W = np.array([
    0.0,
    1000.0,
    4000.0,
    5200.0,
    -1000.0,
])

inverter_test_result = pd.DataFrame(
    calculate_inverter_power(
        ac_electrical_power_W=test_ac_power_W,
        dc_voltage_V=battery_params["nominal_voltage_V"],
    )
)

# ------------------------------------------------------------
# 2. SINGLE-BATTERY-UNIT SCREENING
# Descriptive flags only: this does not simulate BMS intervention.
# Total vehicle auxiliary loads are not included here.
# ------------------------------------------------------------

inverter_test_result["exceeds_unit_continuous_discharge_A"] = (
    inverter_test_result["dc_current_A"]
    > battery_params["continuous_discharge_A"]
)

inverter_test_result["exceeds_unit_continuous_output_W"] = (
    inverter_test_result["dc_power_W"]
    > battery_params["continuous_output_power_W"]
)

inverter_test_result["exceeds_unit_continuous_charge_A"] = (
    -inverter_test_result["dc_current_A"]
    > battery_params["continuous_charge_A"]
)

# ------------------------------------------------------------
# 3. ENGINEERING CHECKS
# ------------------------------------------------------------

assert np.all(
    inverter_test_result["inverter_loss_W"] >= -1e-9
)

assert np.allclose(
    inverter_test_result["dc_power_W"],
    inverter_test_result["ac_electrical_power_W"]
    + inverter_test_result["inverter_loss_W"],
)

assert np.allclose(
    inverter_test_result["dc_power_W"],
    inverter_test_result["dc_voltage_V"]
    * inverter_test_result["dc_current_A"],
)

idle_check = calculate_inverter_power(0.0, 48.0)

assert np.isclose(
    idle_check["dc_power_W"],
    inverter_model_assumptions["enabled_idle_power_W"],
)

regen_check = calculate_inverter_power(-1000.0, 48.0)

assert regen_check["dc_power_W"] < 0
assert regen_check["inverter_loss_W"] >= 0

# ------------------------------------------------------------
# 4. RESULTS
# ------------------------------------------------------------

display(inverter_test_result.round(3))

print("PASS: inverter losses are nonnegative.")
print("PASS: DC-to-AC electrical power balance.")
print("PASS: DC power equals voltage × DC current.")
print("PASS: enabled idle consumption.")
print("PASS: regenerative power direction.")
print("Battery-limit flags are screening results, not BMS actions.")
print("Inverter assumptions are not controller calibration.")

,ac_electrical_power_W,dc_voltage_V,dc_power_W,dc_current_A,inverter_loss_W,exceeds_unit_continuous_discharge_A,exceeds_unit_continuous_output_W,exceeds_unit_continuous_charge_A
0,0.0,48.0,10.000,0.208,10.000,False,False,False
1,1000.0,48.0,1062.632,22.138,62.632,False,False,False
2,4000.0,48.0,4220.526,87.928,220.526,False,False,False
3,5200.0,48.0,5483.684,114.243,283.684,True,True,False
4,-1000.0,48.0,-940.000,-19.583,60.000,False,False,False


PASS: inverter losses are nonnegative.
PASS: DC-to-AC electrical power balance.
PASS: DC power equals voltage × DC current.
PASS: enabled idle consumption.
PASS: regenerative power direction.
Battery-limit flags are screening results, not BMS actions.
Inverter assumptions are not controller calibration.


In [17]:
# STAGE 3 — CELL 11
# Instantaneous battery–inverter coupling for motoring.
# Single-unit scenario; continuous limits only.
# No time integration, regeneration or full BMS protection.

required_objects = [
    "battery_params",
    "battery_model_assumptions",
    "battery_ocv_pack",
    "inverter_model_assumptions",
    "calculate_inverter_power",
    "assumed",
    "np", "pd", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook through Cell 10 first. Missing: "
        + ", ".join(missing_objects)
    )

if (
    assumed["battery_units_in_parallel"] != 1
    or assumed["battery_units_in_series"] != 1
):
    raise RuntimeError(
        "This implementation supports the current single-unit "
        "scenario only."
    )

# New dictionary: supervisory policy assumptions.
# These are not claimed to be actual BMS trip thresholds.
battery_operating_policy = {
    "minimum_discharge_soc": 0.10,
    "evidence": "assumed supervisory operating policy",
}


def couple_battery_inverter_motoring(
    requested_ac_power_W,
    soc,
    polarization_V=0.0,
):
    """
    Solve one instantaneous battery/inverter operating point.

    requested_ac_power_W:
        Nonnegative motor-side electrical power demand.

    soc and polarization_V:
        Battery states supplied to this calculation.
        They are not advanced here.

    Limits:
        Single-unit continuous discharge current and output power.
        Assumed lower SOC boundary.
        Resistance-model power feasibility.

    Excludes:
        Regeneration, actual controller limits, voltage-trip logic,
        temperature protection and short-duration overload operation.
    """
    inputs = np.asarray(
        [requested_ac_power_W, soc, polarization_V],
        dtype=float,
    )

    if not np.all(np.isfinite(inputs)):
        raise ValueError("Inputs must be finite.")

    if requested_ac_power_W < 0:
        raise ValueError(
            "This step supports motoring only. "
            "Regenerative charging will be implemented separately."
        )

    if not 0 <= soc <= 1:
        raise ValueError("SOC must lie between 0 and 1.")

    R0 = float(
        battery_model_assumptions["R0_pack_ohm"]
    )
    eta = float(
        inverter_model_assumptions["motoring_efficiency"]
    )
    idle_W = float(
        inverter_model_assumptions["enabled_idle_power_W"]
    )

    current_limit_A = float(
        battery_params["continuous_discharge_A"]
    )
    power_limit_W = float(
        battery_params["continuous_output_power_W"]
    )
    minimum_soc = float(
        battery_operating_policy["minimum_discharge_soc"]
    )

    if not (
        np.all(np.isfinite([
            R0, eta, idle_W,
            current_limit_A, power_limit_W, minimum_soc,
        ]))
        and R0 > 0
        and 0 < eta <= 1
        and idle_W >= 0
        and current_limit_A > 0
        and power_limit_W > 0
        and 0 <= minimum_soc < 1
    ):
        raise ValueError("Invalid model parameters or policy.")

    ocv_V = float(battery_ocv_pack(soc))
    effective_source_V = ocv_V - polarization_V

    if effective_source_V <= 0:
        raise ValueError("Nonpositive effective battery voltage.")

    # Stay on the lower-current branch of P = E*I - R0*I².
    power_peak_current_A = effective_source_V / (2.0 * R0)

    feasible_current_limit_A = min(
        current_limit_A,
        power_peak_current_A,
    )

    power_at_current_limit_W = (
        effective_source_V
        - R0 * feasible_current_limit_A
    ) * feasible_current_limit_A

    dc_power_cap_W = min(
        power_limit_W,
        power_at_current_limit_W,
    )

    requested_dc_power_W = (
        requested_ac_power_W / eta + idle_W
    )

    # If operation is blocked, disable the modelled inverter.
    # Its disabled consumption is assumed zero in this step.
    if soc <= minimum_soc:
        inverter_enabled = False
        status = "Disabled: assumed lower SOC boundary"
        accepted_ac_power_W = 0.0
        dc_power_W = 0.0

    elif dc_power_cap_W < idle_W:
        inverter_enabled = False
        status = "Disabled: insufficient power for enabled idle load"
        accepted_ac_power_W = 0.0
        dc_power_W = 0.0

    else:
        inverter_enabled = True

        dc_power_W = min(
            requested_dc_power_W,
            dc_power_cap_W,
        )

        accepted_ac_power_W = max(
            0.0,
            eta * (dc_power_W - idle_W),
        )

        status = (
            "Demand limited"
            if requested_dc_power_W > dc_power_cap_W + 1e-9
            else "Demand accepted"
        )

    # Solve battery current on the lower-current branch.
    discriminant = (
        effective_source_V**2 - 4.0 * R0 * dc_power_W
    )

    tolerance = 1e-10 * max(effective_source_V**2, 1.0)

    if discriminant < -tolerance:
        raise RuntimeError("Infeasible battery power request.")

    discriminant = max(discriminant, 0.0)

    battery_current_A = (
        2.0 * dc_power_W
        / (
            effective_source_V
            + np.sqrt(discriminant)
        )
    )

    terminal_voltage_V = (
        effective_source_V - R0 * battery_current_A
    )

    inverter_loss_W = (
        dc_power_W - accepted_ac_power_W
    )

    # Cross-check with the existing inverter function when enabled.
    if inverter_enabled:
        inverter_check = calculate_inverter_power(
            accepted_ac_power_W,
            terminal_voltage_V,
        )

        assert np.isclose(
            inverter_check["dc_power_W"],
            dc_power_W,
            rtol=1e-10,
            atol=1e-8,
        )

    # Engineering checks.
    assert terminal_voltage_V > 0
    assert battery_current_A >= 0
    assert battery_current_A <= current_limit_A + 1e-8
    assert dc_power_W <= power_limit_W + 1e-8
    assert accepted_ac_power_W <= requested_ac_power_W + 1e-8
    assert inverter_loss_W >= -1e-8

    assert np.isclose(
        terminal_voltage_V * battery_current_A,
        dc_power_W,
        rtol=1e-10,
        atol=1e-8,
    )

    assert np.isclose(
        dc_power_W,
        accepted_ac_power_W + inverter_loss_W,
        rtol=1e-10,
        atol=1e-8,
    )

    return {
        "requested_ac_power_W": requested_ac_power_W,
        "accepted_ac_power_W": accepted_ac_power_W,
        "unserved_ac_power_W": max(
            0.0, requested_ac_power_W - accepted_ac_power_W
        ),
        "SOC": soc,
        "polarization_V": polarization_V,
        "ocv_V": ocv_V,
        "terminal_voltage_V": terminal_voltage_V,
        "battery_current_A": battery_current_A,
        "dc_power_W": dc_power_W,
        "inverter_loss_W": inverter_loss_W,
        "inverter_enabled": inverter_enabled,
        "status": status,
    }


# ------------------------------------------------------------
# 1. SYNTHETIC OPERATING-POINT TESTS
# Initial SOC is assumed, not taken from controller BDI.
# ------------------------------------------------------------

test_soc = battery_model_assumptions["initial_soc"]

coupled_operating_points = pd.DataFrame([
    couple_battery_inverter_motoring(
        requested_ac_power_W=demand_W,
        soc=test_soc,
        polarization_V=0.0,
    )
    for demand_W in [0.0, 1000.0, 4000.0, 5200.0, 8000.0]
])

display(
    coupled_operating_points[[
        "requested_ac_power_W",
        "accepted_ac_power_W",
        "unserved_ac_power_W",
        "terminal_voltage_V",
        "battery_current_A",
        "dc_power_W",
        "inverter_loss_W",
        "status",
    ]].round(3)
)

# ------------------------------------------------------------
# 2. TEST THE ASSUMED LOWER-SOC POLICY
# ------------------------------------------------------------

low_soc_check = couple_battery_inverter_motoring(
    requested_ac_power_W=1000.0,
    soc=battery_operating_policy["minimum_discharge_soc"],
    polarization_V=0.0,
)

assert not low_soc_check["inverter_enabled"]
assert np.isclose(low_soc_check["battery_current_A"], 0.0)
assert np.isclose(low_soc_check["accepted_ac_power_W"], 0.0)

assert (
    coupled_operating_points["unserved_ac_power_W"].iloc[-1] > 0
), "The high-demand test should be limited."

print("PASS: battery terminal power equals voltage × current.")
print("PASS: inverter electrical power balance.")
print("PASS: single-unit continuous current and power limits.")
print("PASS: excessive demand is reduced and reported.")
print("PASS: assumed lower-SOC policy disables the inverter.")
print("No time integration or complete BMS/controller protection yet.")

,requested_ac_power_W,accepted_ac_power_W,unserved_ac_power_W,terminal_voltage_V,battery_current_A,dc_power_W,inverter_loss_W,status
0,0.0,0.0,0.0,49.948,0.200,10.000,10.000,Demand accepted
1,1000.0,1000.0,0.0,49.693,21.384,1062.632,62.632,Demand accepted
2,4000.0,4000.0,0.0,48.915,86.284,4220.526,220.526,Demand accepted
3,5200.0,4550.5,649.5,48.769,98.423,4800.000,249.500,Demand limited
4,8000.0,4550.5,3449.5,48.769,98.423,4800.000,249.500,Demand limited


PASS: battery terminal power equals voltage × current.
PASS: inverter electrical power balance.
PASS: single-unit continuous current and power limits.
PASS: excessive demand is reduced and reported.
PASS: assumed lower-SOC policy disables the inverter.
No time integration or complete BMS/controller protection yet.


In [18]:
# STAGE 3 — CELL 12
# Time-domain battery–inverter coupling.
# Synthetic motoring demand; one battery-unit scenario.
#
# Current is recalculated at each numerical step and then held
# during that step. Limits are checked at sampled operating points.
#
# No motor mechanical model, regeneration or complete BMS yet.

required_objects = [
    "battery_params",
    "battery_model_assumptions",
    "battery_operating_policy",
    "couple_battery_inverter_motoring",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook through Cell 11 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. REQUEST PROFILE
# Breakpoints are included exactly in the simulation time grid.
# ------------------------------------------------------------

power_profile_time_s = np.array([
    0.0, 60.0, 180.0, 300.0, 420.0, 540.0, 600.0
])

power_profile_ac_W = np.array([
    0.0, 1000.0, 4000.0, 5200.0, 2000.0, 0.0, 0.0
])

def requested_ac_power_at_time(time_s):
    index = np.searchsorted(
        power_profile_time_s,
        time_s,
        side="right",
    ) - 1

    index = int(np.clip(
        index, 0, len(power_profile_ac_W) - 1
    ))

    return float(power_profile_ac_W[index])

# ------------------------------------------------------------
# 2. COUPLED TIME-DOMAIN SIMULATION
# ------------------------------------------------------------

def simulate_coupled_battery_inverter(max_step_s=0.25):
    """
    Explicit sampled-current integration.

    Records an operating point at every timestamp.
    Current at timestamp k advances states to timestamp k+1.

    Refuses a step crossing the lower SOC boundary rather than
    silently clipping SOC. Boundary-event handling is a later task.
    """
    if not np.isfinite(max_step_s) or max_step_s <= 0:
        raise ValueError("Maximum step must be positive and finite.")

    profile_changes = np.diff(power_profile_time_s)

    if not (
        np.all(profile_changes > 0)
        and len(power_profile_time_s) == len(power_profile_ac_W)
        and np.all(power_profile_ac_W >= 0)
    ):
        raise ValueError("Invalid synthetic power profile.")

    # Build a grid that includes every power-profile transition.
    grid_parts = []

    for start_s, end_s in zip(
        power_profile_time_s[:-1],
        power_profile_time_s[1:],
    ):
        interval_count = int(np.ceil(
            (end_s - start_s) / max_step_s
        ))

        grid_parts.append(
            np.linspace(
                start_s, end_s, interval_count + 1
            )[:-1]
        )

    time_grid_s = np.concatenate(
        grid_parts + [power_profile_time_s[-1:]]
    )

    available_capacity_Ah = (
        battery_params["nominal_capacity_Ah"]
        * battery_model_assumptions[
            "available_capacity_fraction"
        ]
    )

    R1 = float(
        battery_model_assumptions["R1_pack_ohm"]
    )
    C1 = float(
        battery_model_assumptions["C1_pack_F"]
    )

    if not (
        np.isfinite(available_capacity_Ah)
        and available_capacity_Ah > 0
        and np.isfinite(R1)
        and np.isfinite(C1)
        and R1 > 0
        and C1 > 0
    ):
        raise ValueError("Invalid capacity or RC parameters.")

    tau_s = R1 * C1

    soc = float(
        battery_model_assumptions["initial_soc"]
    )
    polarization_V = float(
        battery_model_assumptions["initial_polarization_V"]
    )

    minimum_soc = float(
        battery_operating_policy["minimum_discharge_soc"]
    )

    removed_charge_Ah = 0.0
    records = []

    for k, time_s in enumerate(time_grid_s):
        request_W = requested_ac_power_at_time(time_s)

        operating_point = couple_battery_inverter_motoring(
            requested_ac_power_W=request_W,
            soc=soc,
            polarization_V=polarization_V,
        )

        records.append({
            "time_s": float(time_s),
            **operating_point,
            "net_charge_removed_Ah": removed_charge_Ah,
        })

        # The endpoint is recorded but does not advance the states.
        if k == len(time_grid_s) - 1:
            break

        dt_s = time_grid_s[k + 1] - time_s
        current_A = operating_point["battery_current_A"]

        interval_charge_Ah = current_A * dt_s / 3600.0
        next_soc = (
            soc - interval_charge_Ah / available_capacity_Ah
        )

        if next_soc < minimum_soc:
            raise RuntimeError(
                "A numerical step would cross the lower SOC "
                "boundary. This test requires boundary-event handling "
                "or a shorter experiment; SOC was not clipped."
            )

        decay = np.exp(-dt_s / tau_s)
        one_minus_decay = -np.expm1(-dt_s / tau_s)

        polarization_V = (
            polarization_V * decay
            + current_A * R1 * one_minus_decay
        )

        soc = next_soc
        removed_charge_Ah += interval_charge_Ah

    return pd.DataFrame(records)

# ------------------------------------------------------------
# 3. RUN COARSE AND REFINED SIMULATIONS
# ------------------------------------------------------------

coupled_drivetrain_result = (
    simulate_coupled_battery_inverter(max_step_s=0.25)
)

coupled_drivetrain_refined = (
    simulate_coupled_battery_inverter(max_step_s=0.125)
)

# Despite the table name, only battery/inverter electrical
# coupling is implemented at this point.
result = coupled_drivetrain_result

# ------------------------------------------------------------
# 4. ENGINEERING CHECKS
# ------------------------------------------------------------

assert np.allclose(
    result["terminal_voltage_V"] * result["battery_current_A"],
    result["dc_power_W"],
    atol=1e-8,
)

assert np.allclose(
    result["dc_power_W"],
    result["accepted_ac_power_W"]
    + result["inverter_loss_W"],
    atol=1e-8,
)

assert np.all(
    result["accepted_ac_power_W"]
    <= result["requested_ac_power_W"] + 1e-8
)

assert np.all(
    result["battery_current_A"]
    <= battery_params["continuous_discharge_A"] + 1e-8
)

assert np.all(
    result["dc_power_W"]
    <= battery_params["continuous_output_power_W"] + 1e-8
)

available_capacity_Ah = (
    battery_params["nominal_capacity_Ah"]
    * battery_model_assumptions["available_capacity_fraction"]
)

expected_final_soc = (
    battery_model_assumptions["initial_soc"]
    - result["net_charge_removed_Ah"].iloc[-1]
    / available_capacity_Ah
)

assert np.isclose(
    result["SOC"].iloc[-1],
    expected_final_soc,
    atol=1e-10,
)

# Compare the refined solution at the coarse timestamps.
refined_soc = np.interp(
    result["time_s"],
    coupled_drivetrain_refined["time_s"],
    coupled_drivetrain_refined["SOC"],
)

refined_voltage_V = np.interp(
    result["time_s"],
    coupled_drivetrain_refined["time_s"],
    coupled_drivetrain_refined["terminal_voltage_V"],
)

soc_refinement_error = float(np.max(np.abs(
    result["SOC"].to_numpy() - refined_soc
)))

voltage_refinement_error_V = float(np.max(np.abs(
    result["terminal_voltage_V"].to_numpy()
    - refined_voltage_V
)))

# Numerical screening criteria, not experimental accuracy claims.
assert soc_refinement_error < 1e-4, (
    "SOC changes noticeably with timestep. Refine the simulation."
)

assert voltage_refinement_error_V < 0.02, (
    "Voltage changes noticeably with timestep. Refine the simulation."
)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

coupled_summary = pd.DataFrame([
    (
        "Initial SOC",
        result["SOC"].iloc[0] * 100.0, "%"
    ),
    (
        "Final SOC",
        result["SOC"].iloc[-1] * 100.0, "%"
    ),
    (
        "Charge removed",
        result["net_charge_removed_Ah"].iloc[-1], "Ah"
    ),
    (
        "Minimum terminal voltage",
        result["terminal_voltage_V"].min(), "V"
    ),
    (
        "Maximum accepted battery current",
        result["battery_current_A"].max(), "A"
    ),
    (
        "Maximum unserved motor-side electrical demand",
        result["unserved_ac_power_W"].max(), "W"
    ),
    (
        "Maximum coarse/refined SOC difference",
        soc_refinement_error * 100.0, "percentage points"
    ),
    (
        "Maximum coarse/refined voltage difference",
        voltage_refinement_error_V, "V"
    ),
], columns=["Quantity", "Value", "Unit"])

display(coupled_summary.round({"Value": 6}))

# ------------------------------------------------------------
# 6. PLOTS
# ------------------------------------------------------------

coupled_figure = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[
        "Requested and accepted motor-side electrical power",
        "Battery DC current",
        "Battery terminal voltage",
        "Battery SOC",
    ],
)

for column, label in [
    ("requested_ac_power_W", "Requested AC electrical power"),
    ("accepted_ac_power_W", "Accepted AC electrical power"),
]:
    coupled_figure.add_trace(
        go.Scatter(
            x=result["time_s"],
            y=result[column] / 1000.0,
            name=label,
        ),
        row=1, col=1,
    )

for row_number, column, scale, label in [
    (2, "battery_current_A", 1.0, "Battery current"),
    (3, "terminal_voltage_V", 1.0, "Terminal voltage"),
    (4, "SOC", 100.0, "SOC"),
]:
    coupled_figure.add_trace(
        go.Scatter(
            x=result["time_s"],
            y=result[column] * scale,
            name=label,
        ),
        row=row_number, col=1,
    )

for row_number, unit in [
    (1, "kW"), (2, "A"), (3, "V"), (4, "%")
]:
    coupled_figure.update_yaxes(
        title_text=unit, row=row_number, col=1
    )

coupled_figure.update_xaxes(
    title_text="Simulation time [s]", row=4, col=1
)

coupled_figure.update_layout(
    title="Synthetic battery–inverter demand experiment",
    height=1000,
    template="plotly_white",
    hovermode="x unified",
)

coupled_figure.show()

print("PASS: sampled battery and inverter power balances.")
print("PASS: sampled continuous current and power limits.")
print("PASS: accepted power does not exceed requested power.")
print("PASS: SOC follows accepted battery current.")
print("PASS: timestep-refinement screening.")
print("This is not yet a complete drivetrain or BMS simulation.")

,Quantity,Value,Unit
0,Initial SOC,80.000000,%
1,Final SOC,71.609353,%
2,Charge removed,8.390647,Ah
3,Minimum terminal voltage,48.013007,V
4,Maximum accepted battery current,99.972910,A
5,Maximum unserved motor-side electrical demand,649.500000,W
6,Maximum coarse/refined SOC difference,0.000022,percentage points
7,Maximum coarse/refined voltage difference,0.000002,V


PASS: sampled battery and inverter power balances.
PASS: sampled continuous current and power limits.
PASS: accepted power does not exceed requested power.
PASS: SOC follows accepted battery current.
PASS: timestep-refinement screening.
This is not yet a complete drivetrain or BMS simulation.


In [19]:
# STAGE 4 — CELL 13
# SMAC motor: provisional motoring torque and shaft-power model.
#
# Mechanical relationships only.
# No phase-current dynamics, electrical loss model or regeneration.
# Constant-torque region is an assumption pending curve digitization.

required_objects = [
    "motor_specs",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook's preceding cells first. Missing: "
        + ", ".join(missing_objects)
    )

if (
    motor_specs["manufacturer"] != "Benevelli"
    or motor_specs["model"] != "SMAC 200-035"
):
    raise RuntimeError("The installed SMAC configuration is not selected.")

# ------------------------------------------------------------
# 1. EXPLICIT MOTOR-MODEL ASSUMPTIONS
# Manufacturer specifications are not overwritten.
# ------------------------------------------------------------

motor_model_assumptions = {
    "minimum_model_speed_rpm": 0.0,
    "maximum_model_speed_rpm": 3000.0,
    "constant_torque_region_assumed": True,
    "peak_operation_enabled": False,
    "regeneration_enabled": False,
    "thermal_protection_modelled": False,
    "evidence": (
        "Provisional envelope using nominal manufacturer ratings; "
        "not a digitized torque-speed map"
    ),
}

# ------------------------------------------------------------
# 2. MECHANICAL MOTOR OPERATING POINT
# ------------------------------------------------------------

def calculate_motor_mechanical(
    motor_speed_rpm,
    requested_torque_Nm,
):
    """
    Provisional forward-motoring envelope.

    Speed is supplied externally; this function does not integrate it.
    Requested torque is limited by nominal torque and shaft power.

    Values outside the selected model-speed range are rejected.
    No electrical input power or phase current is calculated.
    """
    speed_rpm, requested_torque = np.broadcast_arrays(
        np.asarray(motor_speed_rpm, dtype=float),
        np.asarray(requested_torque_Nm, dtype=float),
    )

    if not (
        np.all(np.isfinite(speed_rpm))
        and np.all(np.isfinite(requested_torque))
    ):
        raise ValueError("Speed and torque must be finite.")

    if np.any(requested_torque < 0):
        raise ValueError(
            "This step supports nonnegative motoring torque only."
        )

    minimum_rpm = (
        motor_model_assumptions["minimum_model_speed_rpm"]
    )
    maximum_rpm = (
        motor_model_assumptions["maximum_model_speed_rpm"]
    )

    if (
        np.any(speed_rpm < minimum_rpm)
        or np.any(speed_rpm > maximum_rpm)
    ):
        raise ValueError(
            "Speed is outside the provisional 0–3000 rpm range. "
            "This is a model restriction, not a maximum motor rating."
        )

    torque_rating_Nm = float(
        motor_specs["nominal_torque_Nm"]
    )
    power_rating_W = float(
        motor_specs["rated_power_W"]
    )

    if not (
        np.isfinite(torque_rating_Nm)
        and np.isfinite(power_rating_W)
        and torque_rating_Nm > 0
        and power_rating_W > 0
    ):
        raise ValueError("Invalid nominal motor ratings.")

    omega_rad_s = speed_rpm * 2.0 * np.pi / 60.0

    # At zero speed, the shaft-power condition does not restrict
    # torque. Electrical/thermal restrictions still matter physically.
    power_limited_torque_Nm = np.full(
        omega_rad_s.shape,
        np.inf,
        dtype=float,
    )

    np.divide(
        power_rating_W,
        omega_rad_s,
        out=power_limited_torque_Nm,
        where=omega_rad_s > 0,
    )

    torque_limit_Nm = np.minimum(
        torque_rating_Nm,
        power_limited_torque_Nm,
    )

    accepted_torque_Nm = np.minimum(
        requested_torque,
        torque_limit_Nm,
    )

    shaft_power_W = accepted_torque_Nm * omega_rad_s

    return {
        "motor_speed_rpm": speed_rpm,
        "omega_rad_s": omega_rad_s,
        "requested_torque_Nm": requested_torque,
        "torque_limit_Nm": torque_limit_Nm,
        "accepted_torque_Nm": accepted_torque_Nm,
        "unserved_torque_Nm": (
            requested_torque - accepted_torque_Nm
        ),
        "shaft_power_W": shaft_power_W,
    }

# ------------------------------------------------------------
# 3. TEST OPERATING POINTS
# ------------------------------------------------------------

motor_test_result = pd.DataFrame(
    calculate_motor_mechanical(
        motor_speed_rpm=np.array([
            0.0, 500.0, 1000.0, 2000.0, 3000.0
        ]),
        requested_torque_Nm=np.array([
            10.0, 10.0, 17.5, 25.0, 17.5
        ]),
    )
)

display(motor_test_result.round(3))

# ------------------------------------------------------------
# 4. ENGINEERING CHECKS
# ------------------------------------------------------------

assert np.allclose(
    motor_test_result["shaft_power_W"],
    motor_test_result["accepted_torque_Nm"]
    * motor_test_result["omega_rad_s"],
)

assert np.all(
    motor_test_result["accepted_torque_Nm"]
    <= motor_test_result["requested_torque_Nm"] + 1e-9
)

assert np.all(
    motor_test_result["accepted_torque_Nm"]
    <= motor_specs["nominal_torque_Nm"] + 1e-9
)

assert np.all(
    motor_test_result["shaft_power_W"]
    <= motor_specs["rated_power_W"] + 1e-9
)

assert np.isclose(
    motor_test_result.loc[0, "shaft_power_W"],
    0.0,
)

nominal_point = calculate_motor_mechanical(
    motor_specs["nominal_speed_rpm"],
    motor_specs["nominal_torque_Nm"],
)

assert np.isclose(
    nominal_point["shaft_power_W"],
    motor_specs["rated_power_W"],
    rtol=0.02,
)

# ------------------------------------------------------------
# 5. PROVISIONAL ENVELOPE PLOTS
# ------------------------------------------------------------

motor_envelope_result = pd.DataFrame(
    calculate_motor_mechanical(
        motor_speed_rpm=np.linspace(0.0, 3000.0, 151),
        requested_torque_Nm=100.0,
    )
)

from plotly.subplots import make_subplots

motor_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Assumed torque ceiling",
        "Corresponding mechanical shaft power",
    ],
)

motor_figure.add_trace(
    go.Scatter(
        x=motor_envelope_result["motor_speed_rpm"],
        y=motor_envelope_result["torque_limit_Nm"],
        name="Provisional torque ceiling",
    ),
    row=1, col=1,
)

motor_figure.add_trace(
    go.Scatter(
        x=motor_envelope_result["motor_speed_rpm"],
        y=motor_envelope_result["shaft_power_W"] / 1000.0,
        name="Shaft power",
    ),
    row=2, col=1,
)

motor_figure.update_yaxes(
    title_text="Torque [N·m]", row=1, col=1
)
motor_figure.update_yaxes(
    title_text="Power [kW]", row=2, col=1
)
motor_figure.update_xaxes(
    title_text="Motor speed [rpm]", row=2, col=1
)

motor_figure.update_layout(
    title=(
        "SMAC provisional nominal envelope"
        "<br><sup>Constant-torque assumption; "
        "not a digitized manufacturer curve</sup>"
    ),
    template="plotly_white",
    height=650,
)

motor_figure.show()

print("PASS: shaft power equals torque × angular velocity.")
print("PASS: accepted torque respects request and nominal ceiling.")
print("PASS: shaft power respects the selected ceiling.")
print("PASS: zero-speed mechanical power is zero.")
print("PASS: nominal operating point matches the rounded rating.")
print("Motor speed is an input here, not a simulated dynamic state.")
print("Electrical losses, phase currents and thermal limits remain pending.")

,motor_speed_rpm,omega_rad_s,requested_torque_Nm,torque_limit_Nm,accepted_torque_Nm,unserved_torque_Nm,shaft_power_W
0,0.0,0.000,10.0,17.5,10.0,0.0,0.000
1,500.0,52.360,10.0,17.5,10.0,0.0,523.599
2,1000.0,104.720,17.5,17.5,17.5,0.0,1832.596
3,2000.0,209.440,25.0,17.5,17.5,7.5,3665.191
4,3000.0,314.159,17.5,17.5,17.5,0.0,5497.787


PASS: shaft power equals torque × angular velocity.
PASS: accepted torque respects request and nominal ceiling.
PASS: shaft power respects the selected ceiling.
PASS: zero-speed mechanical power is zero.
PASS: nominal operating point matches the rounded rating.
Motor speed is an input here, not a simulated dynamic state.
Electrical losses, phase currents and thermal limits remain pending.


In [20]:
# STAGE 4 — CELL 14
# Provisional motor electrical-to-mechanical loss representation.
#
# Effective assumed loss coefficients:
# NOT an identified SMAC efficiency map.
# Does not estimate phase current or winding temperature.
# Does not enforce battery/inverter power limits yet.

required_objects = [
    "motor_specs",
    "motor_model_assumptions",
    "calculate_motor_mechanical",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook through Cell 13 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. ASSUMED LOSS COEFFICIENTS
# Existing values are retained when rerunning this cell.
# ------------------------------------------------------------

motor_model_assumptions.setdefault(
    "torque_loss_coefficient_W_per_Nm2", 2.0
)

motor_model_assumptions.setdefault(
    "linear_speed_loss_coefficient_W_per_rad_s", 0.05
)

motor_model_assumptions.setdefault(
    "quadratic_speed_loss_coefficient_W_per_rad_s2", 0.001
)

motor_model_assumptions["loss_parameter_evidence"] = (
    "Synthetic effective coefficients; not identified "
    "from motor tests or a manufacturer efficiency map"
)

# ------------------------------------------------------------
# 2. MOTOR ELECTRICAL-DEMAND FUNCTION
# ------------------------------------------------------------

def calculate_motor_electrical_demand(
    motor_speed_rpm,
    requested_torque_Nm,
):
    """
    Forward-motoring electrical power demand.

    Reuses Cell 13's mechanical envelope.
    Losses are calculated from accepted shaft torque.

    Returns motor-side electrical power, NOT battery power.
    No phase current is inferred.
    """
    result = calculate_motor_mechanical(
        motor_speed_rpm=motor_speed_rpm,
        requested_torque_Nm=requested_torque_Nm,
    )

    torque_Nm = np.asarray(
        result["accepted_torque_Nm"], dtype=float
    )
    omega_rad_s = np.asarray(
        result["omega_rad_s"], dtype=float
    )

    a_T = float(
        motor_model_assumptions[
            "torque_loss_coefficient_W_per_Nm2"
        ]
    )
    b_w = float(
        motor_model_assumptions[
            "linear_speed_loss_coefficient_W_per_rad_s"
        ]
    )
    c_w = float(
        motor_model_assumptions[
            "quadratic_speed_loss_coefficient_W_per_rad_s2"
        ]
    )

    if not (
        np.all(np.isfinite([a_T, b_w, c_w]))
        and min(a_T, b_w, c_w) >= 0
    ):
        raise ValueError(
            "Loss coefficients must be finite and nonnegative."
        )

    torque_dependent_loss_W = a_T * np.square(torque_Nm)

    speed_dependent_loss_W = (
        b_w * np.abs(omega_rad_s)
        + c_w * np.square(omega_rad_s)
    )

    motor_loss_W = (
        torque_dependent_loss_W + speed_dependent_loss_W
    )

    shaft_power_W = np.asarray(
        result["shaft_power_W"], dtype=float
    )

    motor_electrical_power_W = (
        shaft_power_W + motor_loss_W
    )

    # Efficiency is undefined when electrical input is zero.
    motor_efficiency = np.full(
        motor_electrical_power_W.shape,
        np.nan,
        dtype=float,
    )

    np.divide(
        shaft_power_W,
        motor_electrical_power_W,
        out=motor_efficiency,
        where=motor_electrical_power_W > 0,
    )

    result.update({
        "torque_dependent_loss_W": torque_dependent_loss_W,
        "speed_dependent_loss_W": speed_dependent_loss_W,
        "motor_loss_W": motor_loss_W,
        "motor_electrical_power_W": motor_electrical_power_W,
        "motor_efficiency": motor_efficiency,
    })

    return result

# ------------------------------------------------------------
# 3. TEST OPERATING POINTS
# ------------------------------------------------------------

motor_electrical_test = pd.DataFrame(
    calculate_motor_electrical_demand(
        motor_speed_rpm=np.array([
            0.0, 0.0, 500.0, 1000.0, 2000.0, 3000.0
        ]),
        requested_torque_Nm=np.array([
            0.0, 10.0, 10.0, 17.5, 17.5, 17.5
        ]),
    )
)

display(
    motor_electrical_test[[
        "motor_speed_rpm",
        "accepted_torque_Nm",
        "shaft_power_W",
        "motor_loss_W",
        "motor_electrical_power_W",
        "motor_efficiency",
    ]].round(4)
)

# ------------------------------------------------------------
# 4. ENGINEERING CHECKS
# ------------------------------------------------------------

assert np.all(
    motor_electrical_test["motor_loss_W"] >= 0
)

assert np.allclose(
    motor_electrical_test["motor_electrical_power_W"],
    motor_electrical_test["shaft_power_W"]
    + motor_electrical_test["motor_loss_W"],
    atol=1e-9,
)

# Holding torque at standstill:
# zero shaft power but positive assumed electrical losses.
holding_check = calculate_motor_electrical_demand(
    motor_speed_rpm=0.0,
    requested_torque_Nm=10.0,
)

assert np.isclose(holding_check["shaft_power_W"], 0.0)
assert holding_check["motor_electrical_power_W"] > 0

# Mathematical zero-torque/zero-speed condition.
zero_check = calculate_motor_electrical_demand(
    motor_speed_rpm=0.0,
    requested_torque_Nm=0.0,
)

assert np.isclose(zero_check["motor_electrical_power_W"], 0.0)
assert np.isnan(zero_check["motor_efficiency"])

defined_efficiencies = (
    motor_electrical_test["motor_efficiency"].dropna()
)

assert np.all(
    (defined_efficiencies >= 0)
    & (defined_efficiencies <= 1)
)

# ------------------------------------------------------------
# 5. SPEED SWEEP AT NOMINAL TORQUE
# This shows demand, not guaranteed available operation.
# ------------------------------------------------------------

motor_loss_sweep = pd.DataFrame(
    calculate_motor_electrical_demand(
        motor_speed_rpm=np.linspace(0.0, 3000.0, 151),
        requested_torque_Nm=motor_specs["nominal_torque_Nm"],
    )
)

motor_loss_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Mechanical output and required motor electrical input",
        "Assumed motor losses",
    ],
)

for column, label in [
    ("shaft_power_W", "Shaft power"),
    ("motor_electrical_power_W", "Motor electrical demand"),
]:
    motor_loss_figure.add_trace(
        go.Scatter(
            x=motor_loss_sweep["motor_speed_rpm"],
            y=motor_loss_sweep[column] / 1000.0,
            name=label,
        ),
        row=1, col=1,
    )

motor_loss_figure.add_trace(
    go.Scatter(
        x=motor_loss_sweep["motor_speed_rpm"],
        y=motor_loss_sweep["motor_loss_W"],
        name="Assumed motor losses",
    ),
    row=2, col=1,
)

motor_loss_figure.update_yaxes(
    title_text="Power [kW]", row=1, col=1
)
motor_loss_figure.update_yaxes(
    title_text="Loss [W]", row=2, col=1
)
motor_loss_figure.update_xaxes(
    title_text="Motor speed [rpm]", row=2, col=1
)

motor_loss_figure.update_layout(
    title=(
        "SMAC development model"
        "<br><sup>Illustrative loss coefficients; "
        "not calibrated motor performance</sup>"
    ),
    template="plotly_white",
    height=650,
)

motor_loss_figure.show()

print("PASS: assumed motor losses are nonnegative.")
print("PASS: electrical input equals shaft output plus losses.")
print("PASS: holding torque can require power at zero speed.")
print("PASS: efficiency is undefined at zero electrical input.")
print("PASS: defined efficiencies lie between zero and one.")
print("Battery/inverter limits are not yet applied to these points.")
print("Motor current and temperature are not calculated by this model.")

,motor_speed_rpm,accepted_torque_Nm,shaft_power_W,motor_loss_W,motor_electrical_power_W,motor_efficiency
0,0.0,0.0,0.0000,0.0000,0.0000,NaN
1,0.0,10.0,0.0000,200.0000,200.0000,0.0000
2,500.0,10.0,523.5988,205.3596,728.9583,0.7183
3,1000.0,17.5,1832.5957,628.7022,2461.2979,0.7446
4,2000.0,17.5,3665.1914,666.8369,4332.0283,0.8461
5,3000.0,17.5,5497.7871,726.9040,6224.6912,0.8832


PASS: assumed motor losses are nonnegative.
PASS: electrical input equals shaft output plus losses.
PASS: holding torque can require power at zero speed.
PASS: efficiency is undefined at zero electrical input.
PASS: defined efficiencies lie between zero and one.
Battery/inverter limits are not yet applied to these points.
Motor current and temperature are not calculated by this model.


In [21]:
# STAGE 4 — CELL 15
# Instantaneous battery–inverter–motor torque limitation.
#
# Reuses existing motor and battery/inverter functions.
# Motor speed remains an externally supplied input.
# No vehicle dynamics, regeneration or full protection model yet.

required_objects = [
    "calculate_motor_mechanical",
    "calculate_motor_electrical_demand",
    "couple_battery_inverter_motoring",
    "battery_model_assumptions",
    "battery_params",
    "motor_specs",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the current notebook through Cell 14 first. Missing: "
        + ", ".join(missing_objects)
    )


def solve_power_limited_motor(
    motor_speed_rpm,
    requested_torque_Nm,
    soc,
    polarization_V=0.0,
):
    """
    Solve a scalar instantaneous motoring operating point.

    Limits requested torque using:
      1. Provisional mechanical motor envelope.
      2. Assumed motor electrical losses.
      3. Averaged inverter demand.
      4. Battery limits implemented in Cell 11.

    States are not advanced here.

    Raises an error if the imposed speed's zero-torque losses
    cannot be supplied. It does not invent a speed trajectory.
    """
    scalar_inputs = [
        motor_speed_rpm,
        requested_torque_Nm,
        soc,
        polarization_V,
    ]

    if any(np.ndim(value) != 0 for value in scalar_inputs):
        raise ValueError(
            "This function accepts scalar operating points only."
        )

    if not np.all(np.isfinite(
        np.asarray(scalar_inputs, dtype=float)
    )):
        raise ValueError("Inputs must be finite.")

    # --------------------------------------------------------
    # 1. MOTOR DEMAND AFTER MECHANICAL ENVELOPE LIMITS
    # --------------------------------------------------------

    requested_motor = calculate_motor_electrical_demand(
        motor_speed_rpm,
        requested_torque_Nm,
    )

    envelope_torque_Nm = float(
        requested_motor["accepted_torque_Nm"]
    )

    motor_power_request_W = float(
        requested_motor["motor_electrical_power_W"]
    )

    initial_supply = couple_battery_inverter_motoring(
        requested_ac_power_W=motor_power_request_W,
        soc=soc,
        polarization_V=polarization_V,
    )

    available_motor_power_W = float(
        initial_supply["accepted_ac_power_W"]
    )

    # --------------------------------------------------------
    # 2. CHECK WHETHER THE IMPOSED SPEED IS FEASIBLE
    # --------------------------------------------------------

    zero_torque_motor = calculate_motor_electrical_demand(
        motor_speed_rpm,
        0.0,
    )

    zero_torque_power_W = float(
        zero_torque_motor["motor_electrical_power_W"]
    )

    if available_motor_power_W < zero_torque_power_W - 1e-8:
        raise RuntimeError(
            "The supplied motor speed is infeasible under the "
            "current model: available electrical power cannot "
            "cover its zero-shaft-torque losses. "
            "A dynamic speed model is required."
        )

    # --------------------------------------------------------
    # 3. SOLVE ACHIEVABLE TORQUE
    # --------------------------------------------------------

    if motor_power_request_W <= available_motor_power_W + 1e-8:
        actual_torque_Nm = envelope_torque_Nm

    else:
        lower_torque_Nm = 0.0
        upper_torque_Nm = envelope_torque_Nm

        # The current loss model has nonnegative coefficients,
        # so electrical demand increases monotonically with
        # nonnegative torque at a fixed nonnegative speed.
        for _ in range(70):
            trial_torque_Nm = (
                lower_torque_Nm + upper_torque_Nm
            ) / 2.0

            trial_motor = calculate_motor_electrical_demand(
                motor_speed_rpm,
                trial_torque_Nm,
            )

            trial_power_W = float(
                trial_motor["motor_electrical_power_W"]
            )

            if trial_power_W <= available_motor_power_W:
                lower_torque_Nm = trial_torque_Nm
            else:
                upper_torque_Nm = trial_torque_Nm

        actual_torque_Nm = lower_torque_Nm

    # --------------------------------------------------------
    # 4. RECALCULATE THE FINAL CONSISTENT OPERATING POINT
    # --------------------------------------------------------

    actual_motor = calculate_motor_electrical_demand(
        motor_speed_rpm,
        actual_torque_Nm,
    )

    actual_motor_power_W = float(
        actual_motor["motor_electrical_power_W"]
    )

    actual_supply = couple_battery_inverter_motoring(
        requested_ac_power_W=actual_motor_power_W,
        soc=soc,
        polarization_V=polarization_V,
    )

    accepted_ac_power_W = float(
        actual_supply["accepted_ac_power_W"]
    )

    shaft_power_W = float(
        actual_motor["shaft_power_W"]
    )

    motor_loss_W = float(
        actual_motor["motor_loss_W"]
    )

    inverter_loss_W = float(
        actual_supply["inverter_loss_W"]
    )

    battery_power_W = float(
        actual_supply["dc_power_W"]
    )

    battery_current_A = float(
        actual_supply["battery_current_A"]
    )

    terminal_voltage_V = float(
        actual_supply["terminal_voltage_V"]
    )

    # --------------------------------------------------------
    # 5. ENGINEERING CHECKS
    # --------------------------------------------------------

    assert np.isclose(
        accepted_ac_power_W,
        actual_motor_power_W,
        rtol=1e-9,
        atol=1e-6,
    ), "Final motor demand is not supplied consistently."

    assert actual_torque_Nm <= envelope_torque_Nm + 1e-8
    assert actual_torque_Nm <= requested_torque_Nm + 1e-8
    assert actual_torque_Nm >= 0

    assert np.isclose(
        battery_power_W,
        shaft_power_W + motor_loss_W + inverter_loss_W,
        rtol=1e-9,
        atol=1e-6,
    ), "Battery-terminal-to-shaft power balance failed."

    assert np.isclose(
        battery_power_W,
        terminal_voltage_V * battery_current_A,
        rtol=1e-9,
        atol=1e-6,
    )

    assert (
        battery_current_A
        <= battery_params["continuous_discharge_A"] + 1e-8
    )

    assert (
        battery_power_W
        <= battery_params["continuous_output_power_W"] + 1e-8
    )

    return {
        "motor_speed_rpm": float(motor_speed_rpm),
        "requested_torque_Nm": float(requested_torque_Nm),
        "envelope_torque_Nm": envelope_torque_Nm,
        "actual_torque_Nm": actual_torque_Nm,
        "unserved_torque_Nm": max(
            0.0,
            requested_torque_Nm - actual_torque_Nm,
        ),
        "shaft_power_W": shaft_power_W,
        "motor_loss_W": motor_loss_W,
        "motor_electrical_power_W": actual_motor_power_W,
        "inverter_loss_W": inverter_loss_W,
        "battery_power_W": battery_power_W,
        "battery_current_A": battery_current_A,
        "terminal_voltage_V": terminal_voltage_V,
        "mechanical_envelope_limited": (
            envelope_torque_Nm < requested_torque_Nm - 1e-6
        ),
        "electrical_power_limited": (
            actual_torque_Nm < envelope_torque_Nm - 1e-6
        ),
    }


# ------------------------------------------------------------
# 6. SYNTHETIC TESTS AT THE ASSUMED INITIAL BATTERY STATE
# ------------------------------------------------------------

test_soc = battery_model_assumptions["initial_soc"]
test_polarization_V = (
    battery_model_assumptions["initial_polarization_V"]
)

test_points = [
    (0.0, 10.0),
    (500.0, 10.0),
    (1000.0, 17.5),
    (2000.0, 25.0),
    (3000.0, 17.5),
]

motor_supply_test = pd.DataFrame([
    solve_power_limited_motor(
        motor_speed_rpm=speed_rpm,
        requested_torque_Nm=torque_Nm,
        soc=test_soc,
        polarization_V=test_polarization_V,
    )
    for speed_rpm, torque_Nm in test_points
])

display(motor_supply_test.round(3))

# The high-speed nominal-torque request should be power-limited
# with the unchanged single-battery scenario and loss assumptions.
nominal_request_check = solve_power_limited_motor(
    motor_speed_rpm=motor_specs["nominal_speed_rpm"],
    requested_torque_Nm=motor_specs["nominal_torque_Nm"],
    soc=test_soc,
    polarization_V=test_polarization_V,
)

assert nominal_request_check["electrical_power_limited"]

# ------------------------------------------------------------
# 7. TORQUE COMPARISON GRAPH
# ------------------------------------------------------------

torque_figure = go.Figure()

torque_figure.add_trace(go.Bar(
    x=[str(int(speed)) for speed, _ in test_points],
    y=motor_supply_test["requested_torque_Nm"],
    name="Requested shaft torque",
))

torque_figure.add_trace(go.Bar(
    x=[str(int(speed)) for speed, _ in test_points],
    y=motor_supply_test["actual_torque_Nm"],
    name="Achievable torque under current model",
))

torque_figure.update_layout(
    title="Synthetic operating points | motor and electrical limits",
    xaxis_title="Externally supplied motor speed [rpm]",
    yaxis_title="Shaft torque [N·m]",
    barmode="group",
    template="plotly_white",
    height=430,
)

torque_figure.show()

print("PASS: actual torque respects request and motor envelope.")
print("PASS: final motor electrical demand is supplied consistently.")
print("PASS: battery terminal power equals current × voltage.")
print("PASS: terminal power equals shaft output plus modelled losses.")
print("PASS: implemented continuous battery limits are respected.")
print("PASS: nominal-speed torque request is electrically limited.")
print("Speed remains an input; this is not vehicle acceleration.")

,motor_speed_rpm,requested_torque_Nm,envelope_torque_Nm,actual_torque_Nm,unserved_torque_Nm,shaft_power_W,motor_loss_W,motor_electrical_power_W,inverter_loss_W,battery_power_W,battery_current_A,terminal_voltage_V,mechanical_envelope_limited,electrical_power_limited
0,0.0,10.0,10.0,10.000,0.000,0.000,200.000,200.000,20.526,220.526,4.420,49.897,False,False
1,500.0,10.0,10.0,10.000,0.000,523.599,205.360,728.958,48.366,777.325,15.621,49.763,False,False
2,1000.0,17.5,17.5,17.500,0.000,1832.596,628.702,2461.298,139.542,2600.840,52.737,49.317,False,False
3,2000.0,25.0,17.5,17.500,7.500,3665.191,666.837,4332.028,238.001,4570.030,93.597,48.827,True,False
4,3000.0,17.5,17.5,13.038,4.462,4096.102,454.398,4550.500,249.500,4800.000,98.423,48.769,False,True


PASS: actual torque respects request and motor envelope.
PASS: final motor electrical demand is supplied consistently.
PASS: battery terminal power equals current × voltage.
PASS: terminal power equals shaft output plus modelled losses.
PASS: implemented continuous battery limits are respected.
PASS: nominal-speed torque request is electrically limited.
Speed remains an input; this is not vehicle acceleration.
